<a href="https://colab.research.google.com/github/ITDeema/Maamar-GP/blob/main/DataSet_Phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set Up

In [ ]:
import os, io, zipfile, requests, pandas as pd, geopandas as gpd
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Maamar"
os.makedirs(f"{BASE}/GBA", exist_ok=True)

# ========== 1) ملف المناخ EPW (الإشعاع + الرياح + الحرارة) ==========
EPW_ZIP = ("https://climate.onebuilding.org/WMO_Region_2_Asia/SAU_Saudi_Arabia/RI_Riyadh/"
           "SAU_RI_Riyadh-Khalid.Intl.AP.404370_TMYx.2011-2025.zip")
r = requests.get(EPW_ZIP, timeout=120)
if r.ok:
    z = zipfile.ZipFile(io.BytesIO(r.content))
    for n in z.namelist():
        if n.endswith(".epw"):
            open(f"{BASE}/{os.path.basename(n)}", "wb").write(z.read(n)); print("✅ EPW:", n)
else:
    print("❌ EPW: الرابط ما اشتغل، حمّليه يدوي من https://climate.onebuilding.org")

# ========== 2) المباني GBA (الأشكال + الارتفاعات) ==========
for t in ["e045_n25_e050_n20", "e045_n30_e050_n25"]:
    r = requests.get(f"https://data.source.coop/tge-labs/globalbuildingatlas-lod1/{t}.parquet", timeout=900)
    open(f"{BASE}/GBA/{t}.parquet", "wb").write(r.content); print("✅ GBA:", t, len(r.content) // 1e6, "MB")

# ========== 3 و4) القطع والشوارع من ArcGIS ==========
ORG = "https://services-ap1.arcgis.com/if9krLUYaMWhxyMO/ArcGIS/rest/services"

def download_layer(url, out):
    parts, off = [], 0
    while True:
        p = {"where": "1=1", "outFields": "*", "returnGeometry": "true", "outSR": 32638,
             "f": "geojson", "resultOffset": off, "resultRecordCount": 2000}
        feats = requests.get(f"{url}/query", params=p, timeout=180).json().get("features", [])
        if not feats: break
        parts.append(gpd.GeoDataFrame.from_features(feats, crs=32638)); off += len(feats)
        print(f"\r{os.path.basename(out)}: {off}", end="")
    gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=32638).to_parquet(out); print(" ✅")

download_layer(f"{ORG}/Riyadh_Parcel/FeatureServer/0", f"{BASE}/riyadh_parcels_all.parquet")

names = [s["name"] for s in requests.get(f"{ORG}?f=json").json().get("services", [])]
street = [n for n in names if "street" in n.lower()]
print("طبقات الشوارع:", street)
if street:
    download_layer(f"{ORG}/{street[0]}/FeatureServer/0", f"{BASE}/riyadh_streets.parquet")

# ========== 5) المحلات (Overture) ==========
!pip -q install overturemaps
!overturemaps download --bbox=46.3,24.4,47.1,25.1 -f geoparquet --type=place -o "{BASE}/overture_places_riyadh.parquet"
print("✅ Overture")

# ========== تأكيد ==========
for f in sorted(os.listdir(BASE)) + [f"GBA/{x}" for x in os.listdir(f"{BASE}/GBA")]:
    print(f)

Mounted at /content/drive
✅ EPW: SAU_RI_Riyadh-Khalid.Intl.AP.404370_TMYx.2011-2025.epw
✅ GBA: e045_n25_e050_n20 132.0 MB
✅ GBA: e045_n30_e050_n25 94.0 MB
riyadh_parcels_all.parquet: 1285086 ✅
طبقات الشوارع: ['Streets_NEW', 'Al_Aqariah_Street']
riyadh_streets.parquet: 144794 ✅
Downloading: 45433 rows [00:02, 16148.44 rows/s]
State saved to /content/drive/MyDrive/Maamar/overture_places_riyadh.parquet.state
✅ Overture
GBA
SAU_RI_Riyadh-Khalid.Intl.AP.404370_TMYx.2011-2025.epw
overture_places_riyadh.parquet
overture_places_riyadh.parquet.state
riyadh_parcels_all.parquet
riyadh_streets.parquet
villas_clean.csv
villas_deema.csv
villas_jana.csv
villas_linah.csv
villas_renad.csv
GBA/e045_n25_e050_n20.parquet
GBA/e045_n30_e050_n25.parquet


# Land Area

In [ ]:
# ===================== ⚙️ الإعدادات: غيّري اسمك هنا بس =====================
STUDENT = "deema"          # deema | renad | jana | linah
# ==========================================================================

import pandas as pd, geopandas as gpd
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Maamar"
IN   = f"{BASE}/villas_{STUDENT}.csv"

d = pd.read_csv(IN)
lat = next(c for c in d.columns if c.lower() in ("lat", "latitude"))
lon = next(c for c in d.columns if c.lower() in ("lon", "lng", "longitude"))

# النقاط (من ملفك) والقطع (من ArcGIS المحمّل)، كلها بالمتر
pts = gpd.GeoDataFrame(d[[]], geometry=gpd.points_from_xy(d[lon], d[lat]), crs=4326).to_crs(32638)
parcels = gpd.read_parquet(f"{BASE}/riyadh_parcels_all.parquet").to_crs(32638)[["ParcelSub", "geometry"]]
parcels["area"] = parcels.geometry.area

# 1) القطعة اللي النقطة داخلها (لو أكثر من وحدة: نفضّل السكنية)
j = gpd.sjoin(pts, parcels, predicate="within", how="left")
j["res"] = j["ParcelSub"].eq("سكني")
j = j.sort_values("res", ascending=False)
area = j[~j.index.duplicated()]["area"]

# 2) النقطة برا أي قطعة: أقرب قطعة خلال 30 م
miss = area.isna()
if miss.any():
    near = gpd.sjoin_nearest(pts[miss], parcels, max_distance=30, how="left")
    area[miss] = near[~near.index.duplicated()]["area"]

d["land_area"] = area.reindex(d.index).round(1)
d.to_csv(IN, index=False, encoding="utf-8-sig")
print(f"✅ {STUDENT}: {d['land_area'].notna().sum()} / {len(d)}")
print("❌ ما لقينا قطعة خلال 30 م:", d["land_area"].isna().sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/geopandas/geodataframe.py:1933: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)


✅ deema: 6141 / 6259
❌ ما لقينا قطعة خلال 30 م: 118


/tmp/ipykernel_592/1598338236.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  area[miss] = near[~near.index.duplicated()]["area"]


In [ ]:
# ===================== ⚙️ الإعدادات: غيّري اسمك هنا بس =====================
STUDENT = "deema"          # deema | renad | jana | linah
# ==========================================================================

import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
IN = f"/content/drive/MyDrive/Maamar/villas_{STUDENT}.csv"

d = pd.read_csv(IN)
before = len(d)

drop = d["land_area"].isna()                                     # ما لقينا لها قطعة
if "area_status" in d.columns:
    drop |= d["area_status"].eq("ما فيه قطعة ضمن 50م")

d = d[~drop].reset_index(drop=True)
d.to_csv(IN, index=False, encoding="utf-8-sig")
print(f"✅ {STUDENT}: انحذف {before - len(d)} صف، وبقى {len(d)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ deema: انحذف 118 صف، وبقى 6141


# building area & main faces




In [ ]:
# ===================== ⚙️ الإعدادات: غيّري اسمك هنا بس =====================
STUDENT = "deema"          # deema | renad | jana | linah
SAVE_EVERY = 200           # يحفظ في ملفك كل كم صف
# ==========================================================================

import numpy as np, pandas as pd, geopandas as gpd, shapely
from shapely.geometry import LineString, Point
from shapely.geometry.polygon import orient
from pyproj import Transformer
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Maamar"

# ======================= البيانات =======================
d = pd.read_csv(f"{BASE}/villas_{STUDENT}.csv")
lat = next(c for c in d.columns if c.lower() in ("lat", "latitude"))
lon = next(c for c in d.columns if c.lower() in ("lon", "lng", "longitude"))

parcels = gpd.read_parquet(f"{BASE}/riyadh_parcels_all.parquet").to_crs(32638)
PA_G = shapely.make_valid(parcels.geometry.to_numpy()); PA_TREE = shapely.STRtree(PA_G)
PA_SUB = parcels["ParcelSub"].to_numpy(object) if "ParcelSub" in parcels else np.array([""] * len(parcels), object)

streets = gpd.read_parquet(f"{BASE}/riyadh_streets.parquet").to_crs(32638)
wcol = next((c for c in streets.columns if "width" in c.lower() or "عرض" in c), None)
ccol = next((c for c in streets.columns if c.upper() in ("SUBTYPECLASS", "CLASS", "STREETCLASS") or "تصنيف" in c), None)
print("أعمدة الشوارع — العرض:", wcol, "| النوع:", ccol)
streets["W"] = pd.to_numeric(streets[wcol], errors="coerce") if wcol else np.nan
streets["W"] = streets["W"].where(streets["W"] > 0)
if ccol: streets["W"] = streets["W"].fillna(streets.groupby(ccol)["W"].transform("median"))
streets["W"] = streets["W"].fillna(streets["W"].median())
ST_G = streets.geometry.to_numpy(); ST_W = streets["W"].to_numpy(float)
ST_C = streets[ccol].astype(str).to_numpy(object) if ccol else np.array([""] * len(streets), object)
ST_TREE = shapely.STRtree(ST_G)

gba = pd.concat([gpd.read_parquet(f"{BASE}/GBA/{t}.parquet")
                 for t in ["e045_n25_e050_n20", "e045_n30_e050_n25"]]).to_crs(32638)
GBA_G = gba.geometry.to_numpy(); GBA_TREE = shapely.STRtree(GBA_G)
TO_M = Transformer.from_crs(4326, 32638, always_xy=True)

# ======================= الثوابت =======================
DIRS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
TOL, MAX_ANGLE, CAND_RADIUS = 3.0, 30, 80          # الشارع الملاصق: حافته ≤3م وموازي ≤30°
NB_SETBACK, COVER_MAX = 1.5, 0.75                  # اشتراطات 4-1-4 و 4-1-1
MIN_AREA = 20.0                                    # أصغر مبنى يُعتبر بيت (م²)

def dir8(az): return DIRS[int(((az + 22.5) % 360) // 45)]
def adiff(a, b): return abs((a - b + 180) % 360 - 180)
def line_dir(line, pt):
    if line.geom_type == "MultiLineString":
        line = min(line.geoms, key=lambda g: g.distance(pt))
    s = line.project(pt)
    a, b = line.interpolate(max(s - 1, 0)), line.interpolate(min(s + 1, line.length))
    return np.degrees(np.arctan2(b.x - a.x, b.y - a.y)) % 180

# ======================= 1) القطعة من الإحداثيات =======================
def get_parcel(la, lo):
    pt = Point(*TO_M.transform(lo, la))
    idx = PA_TREE.query(pt, predicate="intersects")
    if len(idx):
        res = np.array(["سكني" in str(PA_SUB[i]) for i in idx])
        if res.any(): idx = idx[res]                                       # نفضّل السكني مثل كود المساحة
        return PA_G[idx[np.argmin(shapely.area(PA_G[idx]))]]
    idx = PA_TREE.query(pt.buffer(30), predicate="intersects")            # برا: أقرب قطعة خلال 30م
    return PA_G[idx[np.argmin(shapely.distance(PA_G[idx], pt))]] if len(idx) else None

# ======================= 2) أضلاع القطعة وشوارعها =======================
def parcel_sides(P):
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    ring = np.asarray(orient(P, 1.0).exterior.coords); sides = []
    for (x1, y1), (x2, y2) in zip(ring[:-1], ring[1:]):
        dx, dy = x2 - x1, y2 - y1; L = np.hypot(dx, dy)
        if L < 0.5: continue
        seg = LineString([(x1, y1), (x2, y2)]); mid = seg.interpolate(0.5, normalized=True)
        nz = np.degrees(np.arctan2(dy, -dx)) % 360; edir = np.degrees(np.arctan2(dx, dy)) % 180
        street = None
        idx = ST_TREE.query(mid.buffer(CAND_RADIUS))
        if len(idx):
            resid = shapely.distance(ST_G[idx], mid) - ST_W[idx] / 2
            for j in idx[np.argsort(np.abs(resid))]:
                if shapely.distance(ST_G[j], mid) - ST_W[j] / 2 > TOL: break
                par = adiff(line_dir(ST_G[j], mid), edir)
                if min(par, 180 - par) <= MAX_ANGLE: street = j; break
        sides.append(dict(seg=seg, nz=nz, dir=dir8(nz), length=L, street=street,
                          width=ST_W[street] if street is not None else np.nan,
                          cls=ST_C[street] if street is not None else None))
    return P, sides

# ======================= 3أ) البيت الحقيقي (للمبنية) =======================
def real_house(P):
    idx = GBA_TREE.query(P.buffer(5), predicate="intersects")
    if len(idx) == 0: return None
    g = GBA_G[idx]
    pc = P.centroid if P.contains(P.centroid) else P.representative_point()
    area = shapely.area(g); inter = shapely.area(shapely.intersection(g, P))
    frac = np.divide(inter, area, out=np.zeros_like(area), where=area > 0)
    site = shapely.within(shapely.centroid(g), P) | (frac >= 0.5) | shapely.contains(g, pc) \
           | (shapely.distance(g, pc) <= 2.0)
    if not site.any(): return None
    H = shapely.union_all(g[site]).intersection(P.buffer(1.0))
    if H.geom_type == "MultiPolygon": H = max(H.geoms, key=lambda q: q.area)
    return H if (not H.is_empty and H.area >= MIN_AREA) else None

# ======================= 3ب) الغلاف (للفاضية) =======================
def setback(width, cls, is_main):
    """اشتراطات 4-1-4 و 4-1-5 (صفحة 17)"""
    cls = cls if isinstance(cls, str) else ""
    w = width if pd.notna(width) and width > 0 else 0
    if "ممر" in cls:                  return 1.5
    if "ساح" in cls or "ميد" in cls:  return max(3.0, w / 5)
    if is_main:                        return 6.0 if w >= 30 else max(3.0, w / 5)
    return max(2.0, w / 5)

FALLBACK_R = 50.0                                   # أبعد مسافة نبحث فيها عن شارع قدّام الضلع (م)
def nearest_front_street(sides):
    """لو ما فيه شارع ملاصق: نختار الضلع اللي قدّامه أقرب شارع (حافة الشارع، مو خط المنتصف)"""
    best = None
    for s in sides:
        mid = s["seg"].interpolate(0.5, normalized=True)
        nx, ny = np.sin(np.radians(s["nz"])), np.cos(np.radians(s["nz"]))
        idx = ST_TREE.query(mid.buffer(FALLBACK_R))
        for j in idx:
            q = ST_G[j].interpolate(ST_G[j].project(mid))               # أقرب نقطة على الشارع
            vx, vy = q.x - mid.x, q.y - mid.y; dist = np.hypot(vx, vy)
            if dist > 0 and (vx * nx + vy * ny) / dist < np.cos(np.radians(60)):
                continue                                                # الشارع مو قدّام الضلع
            edge = max(dist - ST_W[j] / 2, 0)
            if edge <= FALLBACK_R and (best is None or edge < best[0]):
                best = (edge, s, j)
    if best is None: return False
    _, s, j = best
    s.update(street=j, width=ST_W[j], cls=ST_C[j])
    return True

def envelope(P, sides):
    st = [s for s in sides if s["street"] is not None]
    if not st and nearest_front_street(sides):                          # تصحيح: أقرب شارع قدّام القطعة
        st = [s for s in sides if s["street"] is not None]
    main = max(st, key=lambda s: (s["width"], s["length"])) if st else None
    cut, nb = [], []
    for s in sides:
        if s["street"] is not None:
            s["setback"] = setback(s["width"], s["cls"], s is main)
            s["type"] = "main street" if s is main else "street"
        else:
            s["setback"], s["type"] = NB_SETBACK, "neighbour"; nb.append(s["seg"])
        cut.append(s["seg"].buffer(s["setback"], cap_style=3))
    E = P.difference(shapely.union_all(cut))
    if E.geom_type == "MultiPolygon": E = max(E.geoms, key=lambda g: g.area)
    if E.is_empty or E.area < 0.2 * P.area: E = P
    if E.area > COVER_MAX * P.area and nb:                              # > 75%: نزيد من جهة الجيران
        nbu = shapely.union_all(nb); lo, hi = NB_SETBACK, NB_SETBACK + 10
        for _ in range(25):
            mid = (lo + hi) / 2
            Em = E.difference(nbu.buffer(mid, cap_style=3))
            Em = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
            lo, hi = (mid, hi) if Em.area > COVER_MAX * P.area else (lo, mid)
        Em = E.difference(nbu.buffer(hi, cap_style=3))
        E = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
        for s in sides:
            if s["type"] == "neighbour": s["setback"] = round(hi, 2)
    return E, main

# ======================= 4) الحساب + الحفظ المرحلي =======================
import os
FILE = f"{BASE}/villas_{STUDENT}.csv"
NEW = ["building_area", "main_street_dir", "bldg_status"]
for c in NEW:
    if c not in d: d[c] = pd.Series(np.nan, index=d.index, dtype=object if c != "building_area" else float)

def save():
    tmp = FILE + ".tmp"
    d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)   # ما ينخرب الملف لو انقطع

todo = d.index[d["bldg_status"].isna() | (d["bldg_status"] == "ما فيه شارع ملاصق")]   # يكمّل من وين وقف + يعيد اللي ما لقى شارع
print(f"باقي {len(todo)} من {len(d)} صف")
for n, i in enumerate(todo, 1):
    try:
        P = get_parcel(d.at[i, lat], d.at[i, lon])
        if P is None:
            d.at[i, "bldg_status"] = "ما فيه قطعة"
        else:
            P, sides = parcel_sides(P)
            E_env, main = envelope(P, sides)
            H = real_house(P)
            E = H if H is not None else E_env                            # مبني: شكله الحقيقي / فاضي: الغلاف
            d.at[i, "building_area"] = round(E.area, 1)
            d.at[i, "main_street_dir"] = main["dir"] if main else np.nan
            d.at[i, "bldg_status"] = "ok" if main else "ما فيه شارع ملاصق"
    except Exception as e:
        d.at[i, "bldg_status"] = f"خطأ: {type(e).__name__}"
    if n % SAVE_EVERY == 0:
        save(); print(f"  حُفظ {n}/{len(todo)}")
save()

print("\n✅ خلص")
print(d["bldg_status"].value_counts().to_string())
print("\nالواجهة الرئيسية:"); print(d["main_street_dir"].value_counts().to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
أعمدة الشوارع — العرض: STREETWIDTH | النوع: STREETCLASS
باقي 145 من 6141 صف

✅ خلص
bldg_status
ok                   6140
ما فيه شارع ملاصق       1

الواجهة الرئيسية:
main_street_dir
NW    1321
SE    1310
SW    1208
NE    1143
N      339
S      315
W      270
E      234


In [ ]:
# ===================== ⚙️ غيّري اسمك هنا بس =====================
STUDENT = "deema"          # deema | renad | jana | linah
# ===============================================================

import os, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
FILE = f"/content/drive/MyDrive/Maamar/villas_{STUDENT}.csv"

d = pd.read_csv(FILE)
print("قبل:", len(d), "صف")
print(d["bldg_status"].value_counts(dropna=False).to_string())

# 1) نحذف الصفوف: ما فيه شارع ملاصق + أي خطأ
bad = (d["bldg_status"] == "ما فيه شارع ملاصق") | d["bldg_status"].astype(str).str.startswith("خطأ")
d = d[~bad]

# 2) نحذف العمودين
d = d.drop(columns=["area_status", "bldg_status"], errors="ignore")

tmp = FILE + ".tmp"
d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)
print(f"\n✅ انحذف {bad.sum()} صف — بقى {len(d)} صف")
print("الأعمدة:", list(d.columns))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
قبل: 6141 صف
bldg_status
ok                   6140
ما فيه شارع ملاصق       1

✅ انحذف 1 صف — بقى 6140 صف
الأعمدة: ['lat', 'lon', 'landuse', 'land_area', 'parcel_wkt', 'building_area', 'main_street_dir']


#  building height

In [ ]:
# ===================== ⚙️ الإعدادات: غيّري اسمك هنا بس =====================
STUDENT = "deema"          # deema | renad | jana | linah
SAVE_EVERY = 200           # يحفظ في ملفك كل كم صف
# ==========================================================================

import os, numpy as np, pandas as pd, geopandas as gpd, shapely
from shapely.geometry import Point
from pyproj import Transformer
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Maamar"
FILE = f"{BASE}/villas_{STUDENT}.csv"

# ======================= البيانات =======================
d = pd.read_csv(FILE)
lat = next(c for c in d.columns if c.lower() in ("lat", "latitude"))
lon = next(c for c in d.columns if c.lower() in ("lon", "lng", "longitude"))

parcels = gpd.read_parquet(f"{BASE}/riyadh_parcels_all.parquet").to_crs(32638)
PA_G = shapely.make_valid(parcels.geometry.to_numpy()); PA_TREE = shapely.STRtree(PA_G)
PA_SUB = parcels["ParcelSub"].to_numpy(object) if "ParcelSub" in parcels else np.array([""] * len(parcels), object)

gba = pd.concat([gpd.read_parquet(f"{BASE}/GBA/{t}.parquet")
                 for t in ["e045_n25_e050_n20", "e045_n30_e050_n25"]]).to_crs(32638)
hcol = next(c for c in gba.columns if "height" in c.lower())
print("عمود الارتفاع في GBA:", hcol)
GBA_G = gba.geometry.to_numpy(); GBA_TREE = shapely.STRtree(GBA_G)
GBA_H = pd.to_numeric(gba[hcol], errors="coerce").to_numpy(float)
TO_M = Transformer.from_crs(4326, 32638, always_xy=True)

MIN_AREA = 20.0     # أصغر مبنى يُعتبر بيت (م²) — نفس كود الداتا سيت
MAX_H    = 14.0     # للقطع الفاضية بس — اشتراطات 5-1-5 (المبنية ناخذ ارتفاعها الحقيقي حتى لو أعلى)

# ======================= 1) القطعة (نفس كود الداتا سيت) =======================
def get_parcel(la, lo):
    pt = Point(*TO_M.transform(lo, la))
    idx = PA_TREE.query(pt, predicate="intersects")
    if len(idx):
        res = np.array(["سكني" in str(PA_SUB[i]) for i in idx])
        if res.any(): idx = idx[res]
        return PA_G[idx[np.argmin(shapely.area(PA_G[idx]))]]
    idx = PA_TREE.query(pt.buffer(30), predicate="intersects")
    return PA_G[idx[np.argmin(shapely.distance(PA_G[idx], pt))]] if len(idx) else None

# ======================= 2) البيت الحقيقي + ارتفاعه =======================
def house_height(P):
    """نفس اختيار البيت في كود الداتا سيت، ويرجّع ارتفاعه (أطول مبنى في البيت) — أو None لو القطعة فاضية"""
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    idx = GBA_TREE.query(P.buffer(5), predicate="intersects")
    if len(idx) == 0: return None
    g = GBA_G[idx]; h = GBA_H[idx]
    pc = P.centroid if P.contains(P.centroid) else P.representative_point()
    area = shapely.area(g); inter = shapely.area(shapely.intersection(g, P))
    frac = np.divide(inter, area, out=np.zeros_like(area), where=area > 0)
    site = shapely.within(shapely.centroid(g), P) | (frac >= 0.5) | shapely.contains(g, pc) \
           | (shapely.distance(g, pc) <= 2.0)
    if not site.any(): return None
    H = shapely.union_all(g[site]).intersection(P.buffer(1.0))
    if H.geom_type == "MultiPolygon": H = max(H.geoms, key=lambda q: q.area)
    if H.is_empty or H.area < MIN_AREA: return None                     # فاضية
    w = shapely.area(shapely.intersection(g[site], H))                  # كم من كل مبنى داخل البيت
    ok = (w > 0) & np.isfinite(h[site]) & (h[site] > 0)
    if not ok.any(): return np.nan                                      # مبنية بس GBA ما فيه ارتفاع
    return float(h[site][ok].max())                                    # الأطول

# ======================= 3) الحساب + الحفظ المرحلي =======================
for c in ["building_height", "height_source"]:
    if c not in d: d[c] = pd.Series(np.nan, index=d.index, dtype=float if c == "building_height" else object)

def save():
    tmp = FILE + ".tmp"
    d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)

todo = d.index[d["building_height"].isna()]                              # يكمّل من وين وقف
print(f"باقي {len(todo)} من {len(d)} صف")
for n, i in enumerate(todo, 1):
    try:
        P = get_parcel(d.at[i, lat], d.at[i, lon])
        h = house_height(P) if P is not None else None
        if h is None:                     src, h = "vacant: max 14", MAX_H          # فاضية → الحد النظامي
        elif np.isnan(h):                 src, h = "GBA no height: max 14", MAX_H   # مبنية بدون ارتفاع
        else:                             src = "GBA"
        d.at[i, "building_height"] = round(h, 1); d.at[i, "height_source"] = src
    except Exception as e:
        d.at[i, "height_source"] = f"خطأ: {type(e).__name__}"
    if n % SAVE_EVERY == 0:
        save(); print(f"  حُفظ {n}/{len(todo)}")
save()

print("\n✅ خلص")
print(d["height_source"].value_counts().to_string())
print("\nالارتفاع:"); print(d["building_height"].describe().round(1).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
عمود الارتفاع في GBA: height
باقي 6140 من 6140 صف
  حُفظ 200/6140
  حُفظ 400/6140
  حُفظ 600/6140
  حُفظ 800/6140
  حُفظ 1000/6140
  حُفظ 1200/6140
  حُفظ 1400/6140
  حُفظ 1600/6140
  حُفظ 1800/6140
  حُفظ 2000/6140
  حُفظ 2200/6140
  حُفظ 2400/6140
  حُفظ 2600/6140
  حُفظ 2800/6140
  حُفظ 3000/6140
  حُفظ 3200/6140
  حُفظ 3400/6140
  حُفظ 3600/6140
  حُفظ 3800/6140
  حُفظ 4000/6140
  حُفظ 4200/6140
  حُفظ 4400/6140
  حُفظ 4600/6140
  حُفظ 4800/6140
  حُفظ 5000/6140
  حُفظ 5200/6140
  حُفظ 5400/6140
  حُفظ 5600/6140
  حُفظ 5800/6140
  حُفظ 6000/6140

✅ خلص
height_source
GBA               4155
vacant: max 14    1985

الارتفاع:
count    6140.0
mean        9.0
std         4.1
min         0.0
25%         5.8
50%         8.0
75%        14.0
max        20.2


In [ ]:
STUDENT = "deema"
import os, pandas as pd
FILE = f"/content/drive/MyDrive/Maamar/villas_{STUDENT}.csv"
d = pd.read_csv(FILE)
before = len(d)
d = d[~(d["building_height"] < 3)]
tmp = FILE + ".tmp"; d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)
print(f"✅ انحذف {before - len(d)} صف — بقى {len(d)}")

✅ انحذف 349 صف — بقى 5791


## radiation include (SVF & Horizon angle)

In [ ]:
!pip install pvlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 56.0 MB/s  0:00:00


In [ ]:
# ===================== ⚙️ الإعدادات: غيّري اسمك هنا بس =====================
STUDENT = "deema"          # deema | renad | jana | linah
SAVE_EVERY = 50            # يحفظ في ملفك كل كم صف
# ==========================================================================
# الإشعاع على الواجهات: Rad صيف + شتاء، و Rad_block صيف — لكل جهة من 8 (24 عمود)
# المبنية: شكل البيت من GBA / الفاضية: الغلاف النظامي — والارتفاع من building_height

import os, glob, zipfile, time, numpy as np, pandas as pd, geopandas as gpd, shapely, pvlib
from shapely.geometry import LineString, Point
from shapely.geometry.polygon import orient
from pyproj import Transformer
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Maamar"
FILE = f"{BASE}/villas_{STUDENT}.csv"

# ======================= الثوابت =======================
RES     = 2.0     # شبكة النقاط على الجدار (م) — Chatzipoulka et al. 2018
NSEC    = 72      # 72 قطاع × 5° — shadow package (res_angle = 5)
SEARCH  = 100.0   # نبحث عن الجيران لين 100 م من النقطة
OFFSET  = 0.10    # النقطة برا الجدار بـ10 سم عشان الجدار ما يحجب نفسه
MIN_AREA = 20.0
DIRS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
SEASONS = {"Summer": ([6, 7, 8], 92), "Winter": ([12, 1, 2], 90)}
SEC_C = (np.arange(NSEC) + 0.5) * 360 / NSEC                     # منتصف كل قطاع
def sec_of(az): return (np.asarray(az) // (360 / NSEC)).astype(int) % NSEC
def dir8(az):   return DIRS[int(((az + 22.5) % 360) // 45)]
def adiff(a, b): return np.abs((a - b + 180) % 360 - 180)
TOL, MAX_ANGLE, CAND_RADIUS = 3.0, 30, 80          # الشارع الملاصق: حافته ≤3م وموازي ≤30°
NB_SETBACK, COVER_MAX = 1.5, 0.75                  # اشتراطات 4-1-4 و 4-1-1
VACANT_H = 14.0                                    # ارتفاع الفاضية (نفس كود الارتفاع)
def line_dir(line, pt):
    if line.geom_type == "MultiLineString":
        line = min(line.geoms, key=lambda g: g.distance(pt))
    s = line.project(pt)
    a, b = line.interpolate(max(s - 1, 0)), line.interpolate(min(s + 1, line.length))
    return np.degrees(np.arctan2(b.x - a.x, b.y - a.y)) % 180

def house_axis(E):
    """اتجاه البيت العام = أصغر مستطيل يحيط فيه (SMBR) — Duchêne et al. 2003"""
    R = np.asarray(shapely.minimum_rotated_rectangle(E).exterior.coords)
    (x0, y0), (x1, y1) = R[0], R[1]
    return np.degrees(np.arctan2(x1 - x0, y1 - y0)) % 90                 # محاور البيت تتكرر كل 90°

def wall_dir(nz, axis):
    """الجهة (8) لكل جدار بعد ما نربطه بمحاور البيت: محور البيت + مضاعفات 45°
       (الجدران المتوازية تاخذ نفس الجهة؛ الحساب الفيزيائي يبقى على زاوية الجدار الحقيقية)"""
    snapped = axis + np.round((np.asarray(nz) - axis) / 45) * 45
    return [dir8(a % 360) for a in snapped]

# ======================= البيانات =======================
d = pd.read_csv(FILE)
lat = next(c for c in d.columns if c.lower() in ("lat", "latitude"))
lon = next(c for c in d.columns if c.lower() in ("lon", "lng", "longitude"))
assert "building_height" in d, "شغّلي كود الارتفاع أول"

parcels = gpd.read_parquet(f"{BASE}/riyadh_parcels_all.parquet").to_crs(32638)
PA_G = shapely.make_valid(parcels.geometry.to_numpy()); PA_TREE = shapely.STRtree(PA_G)
PA_SUB = parcels["ParcelSub"].to_numpy(object) if "ParcelSub" in parcels else np.array([""] * len(parcels), object)

streets = gpd.read_parquet(f"{BASE}/riyadh_streets.parquet").to_crs(32638)
wcol = next((c for c in streets.columns if "width" in c.lower() or "عرض" in c), None)
ccol = next((c for c in streets.columns if c.upper() in ("SUBTYPECLASS", "CLASS", "STREETCLASS") or "تصنيف" in c), None)
print("أعمدة الشوارع — العرض:", wcol, "| النوع:", ccol)
streets["W"] = pd.to_numeric(streets[wcol], errors="coerce") if wcol else np.nan
streets["W"] = streets["W"].where(streets["W"] > 0)
if ccol: streets["W"] = streets["W"].fillna(streets.groupby(ccol)["W"].transform("median"))
streets["W"] = streets["W"].fillna(streets["W"].median())
ST_G = streets.geometry.to_numpy(); ST_W = streets["W"].to_numpy(float)
ST_C = streets[ccol].astype(str).to_numpy(object) if ccol else np.array([""] * len(streets), object)
ST_TREE = shapely.STRtree(ST_G)

gba = pd.concat([gpd.read_parquet(f"{BASE}/GBA/{t}.parquet")
                 for t in ["e045_n25_e050_n20", "e045_n30_e050_n25"]]).to_crs(32638)
hcol = next(c for c in gba.columns if "height" in c.lower())
GBA_G = gba.geometry.to_numpy(); GBA_TREE = shapely.STRtree(GBA_G)
GBA_H = pd.to_numeric(gba[hcol], errors="coerce").to_numpy(float)
TO_M = Transformer.from_crs(4326, 32638, always_xy=True)

# ---------- EPW: مطار الملك خالد TMYx ----------
epw_files = glob.glob(f"{BASE}/**/*.epw", recursive=True)
if not epw_files:
    z = glob.glob(f"{BASE}/**/*Riyadh*Khalid*.zip", recursive=True)[0]
    zipfile.ZipFile(z).extractall("/content/epw"); epw_files = glob.glob("/content/epw/**/*.epw", recursive=True)
EPW_FILE = next((f for f in epw_files if "Khalid" in f), epw_files[0]); print("EPW:", os.path.basename(EPW_FILE))
epw, meta = pvlib.iotools.read_epw(EPW_FILE)
# التوقيت: اختبار الإغلاق GHI = DNI·sin(alt) + DHI على نفس الملف → أقل خطأ عند الإزاحة 0
# (RMSE = 16.5 W/m²؛ +30 دقيقة = 42.7) → مكان الشمس عند epw.index نفسه
tm = epw.index
sp = pvlib.solarposition.get_solarposition(tm, meta["latitude"], meta["longitude"])   # NREL SPA
ALT, AZ = sp["apparent_elevation"].to_numpy(), sp["azimuth"].to_numpy()
DNI, DHI = epw["dni"].to_numpy(float), epw["dhi"].to_numpy(float)
SEA = {}
for s, (ms, days) in SEASONS.items():
    m = np.isin(tm.month, ms)
    SEA[s] = dict(alt=ALT[m], az=AZ[m], dni=DNI[m], dhi=DHI[m], days=days)
    print(f"{s}: {m.sum()} ساعة")

# ======================= 1) القطعة + البيت =======================
def get_parcel(la, lo):
    pt = Point(*TO_M.transform(lo, la))
    idx = PA_TREE.query(pt, predicate="intersects")
    if len(idx):
        res = np.array(["سكني" in str(PA_SUB[i]) for i in idx])
        if res.any(): idx = idx[res]
        return PA_G[idx[np.argmin(shapely.area(PA_G[idx]))]]
    idx = PA_TREE.query(pt.buffer(30), predicate="intersects")
    return PA_G[idx[np.argmin(shapely.distance(PA_G[idx], pt))]] if len(idx) else None

def real_house(P):
    """نفس كود الداتا سيت — ويرجّع أرقام مباني GBA اللي تكوّن البيت (عشان ما نحسبها جيران)"""
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    idx = GBA_TREE.query(P.buffer(5), predicate="intersects")
    if len(idx) == 0: return None, None
    g = GBA_G[idx]
    pc = P.centroid if P.contains(P.centroid) else P.representative_point()
    area = shapely.area(g); inter = shapely.area(shapely.intersection(g, P))
    frac = np.divide(inter, area, out=np.zeros_like(area), where=area > 0)
    site = shapely.within(shapely.centroid(g), P) | (frac >= 0.5) | shapely.contains(g, pc) \
           | (shapely.distance(g, pc) <= 2.0)
    if not site.any(): return None, None
    H = shapely.union_all(g[site]).intersection(P.buffer(1.0))
    if H.geom_type == "MultiPolygon": H = max(H.geoms, key=lambda q: q.area)
    return (H, idx[site]) if (not H.is_empty and H.area >= MIN_AREA) else (None, None)

# ======================= 1ب) أضلاع القطعة وشوارعها =======================
def parcel_sides(P):
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    ring = np.asarray(orient(P, 1.0).exterior.coords); sides = []
    for (x1, y1), (x2, y2) in zip(ring[:-1], ring[1:]):
        dx, dy = x2 - x1, y2 - y1; L = np.hypot(dx, dy)
        if L < 0.5: continue
        seg = LineString([(x1, y1), (x2, y2)]); mid = seg.interpolate(0.5, normalized=True)
        nz = np.degrees(np.arctan2(dy, -dx)) % 360; edir = np.degrees(np.arctan2(dx, dy)) % 180
        street = None
        idx = ST_TREE.query(mid.buffer(CAND_RADIUS))
        if len(idx):
            resid = shapely.distance(ST_G[idx], mid) - ST_W[idx] / 2
            for j in idx[np.argsort(np.abs(resid))]:
                if shapely.distance(ST_G[j], mid) - ST_W[j] / 2 > TOL: break
                par = adiff(line_dir(ST_G[j], mid), edir)
                if min(par, 180 - par) <= MAX_ANGLE: street = j; break
        sides.append(dict(seg=seg, nz=nz, dir=dir8(nz), length=L, street=street,
                          width=ST_W[street] if street is not None else np.nan,
                          cls=ST_C[street] if street is not None else None))
    return P, sides

# ======================= 1ج) الغلاف (للفاضية) =======================
def setback(width, cls, is_main):
    """اشتراطات 4-1-4 و 4-1-5 (صفحة 17)"""
    cls = cls if isinstance(cls, str) else ""
    w = width if pd.notna(width) and width > 0 else 0
    if "ممر" in cls:                  return 1.5
    if "ساح" in cls or "ميد" in cls:  return max(3.0, w / 5)
    if is_main:                        return 6.0 if w >= 30 else max(3.0, w / 5)
    return max(2.0, w / 5)

FALLBACK_R = 50.0                                   # أبعد مسافة نبحث فيها عن شارع قدّام الضلع (م)
def nearest_front_street(sides):
    """لو ما فيه شارع ملاصق: نختار الضلع اللي قدّامه أقرب شارع (حافة الشارع، مو خط المنتصف)"""
    best = None
    for s in sides:
        mid = s["seg"].interpolate(0.5, normalized=True)
        nx, ny = np.sin(np.radians(s["nz"])), np.cos(np.radians(s["nz"]))
        idx = ST_TREE.query(mid.buffer(FALLBACK_R))
        for j in idx:
            q = ST_G[j].interpolate(ST_G[j].project(mid))               # أقرب نقطة على الشارع
            vx, vy = q.x - mid.x, q.y - mid.y; dist = np.hypot(vx, vy)
            if dist > 0 and (vx * nx + vy * ny) / dist < np.cos(np.radians(60)):
                continue                                                # الشارع مو قدّام الضلع
            edge = max(dist - ST_W[j] / 2, 0)
            if edge <= FALLBACK_R and (best is None or edge < best[0]):
                best = (edge, s, j)
    if best is None: return False
    _, s, j = best
    s.update(street=j, width=ST_W[j], cls=ST_C[j])
    return True

def envelope(P, sides):
    st = [s for s in sides if s["street"] is not None]
    if not st and nearest_front_street(sides):                          # تصحيح: أقرب شارع قدّام القطعة
        st = [s for s in sides if s["street"] is not None]
    main = max(st, key=lambda s: (s["width"], s["length"])) if st else None
    cut, nb = [], []
    for s in sides:
        if s["street"] is not None:
            s["setback"] = setback(s["width"], s["cls"], s is main)
            s["type"] = "main street" if s is main else "street"
        else:
            s["setback"], s["type"] = NB_SETBACK, "neighbour"; nb.append(s["seg"])
        cut.append(s["seg"].buffer(s["setback"], cap_style=3))
    E = P.difference(shapely.union_all(cut))
    if E.geom_type == "MultiPolygon": E = max(E.geoms, key=lambda g: g.area)
    if E.is_empty or E.area < 0.2 * P.area: E = P
    if E.area > COVER_MAX * P.area and nb:                              # > 75%: نزيد من جهة الجيران
        nbu = shapely.union_all(nb); lo, hi = NB_SETBACK, NB_SETBACK + 10
        for _ in range(25):
            mid = (lo + hi) / 2
            Em = E.difference(nbu.buffer(mid, cap_style=3))
            Em = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
            lo, hi = (mid, hi) if Em.area > COVER_MAX * P.area else (lo, mid)
        Em = E.difference(nbu.buffer(hi, cap_style=3))
        E = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
        for s in sides:
            if s["type"] == "neighbour": s["setback"] = round(hi, 2)
    return E, main

# ======================= 2) نقاط الواجهات =======================
def facade_points(E, Hb):
    """كل جدار: مربعات 2×2 م والنقطة في نص المربع، برا الجدار بـ10 سم"""
    hs = np.arange(RES / 2, Hb, RES) if Hb > RES else np.array([Hb / 2])
    ring = np.asarray(orient(E, 1.0).exterior.coords)
    X, Y, Z, NZ, S, W = [], [], [], [], [], []; s0 = 0.0
    for k, ((x1, y1), (x2, y2)) in enumerate(zip(ring[:-1], ring[1:])):
        dx, dy = x2 - x1, y2 - y1; L = np.hypot(dx, dy)
        if L < 1.0: s0 += L; continue
        nx, ny = dy / L, -dx / L                                          # العمود للخارج
        n = max(1, int(round(L / RES))); t = (np.arange(n) + 0.5) / n
        px, py = x1 + t * dx + nx * OFFSET, y1 + t * dy + ny * OFFSET
        nz = np.degrees(np.arctan2(nx, ny)) % 360
        for h in hs:
            X.append(px); Y.append(py); Z.append(np.full(n, h)); NZ.append(np.full(n, nz))
            S.append(s0 + t * L); W.append(np.full(n, k))
        s0 += L
    c = np.concatenate
    return pd.DataFrame(dict(x=c(X), y=c(Y), h=c(Z), nz=c(NZ), s=c(S), wall=c(W)))

# ======================= 3) زاوية الأفق =======================
def obstacles(E, Hb, site_idx):
    """حدود الجيران (ضمن 100م) + بيتنا نفسه (عشان البيت اللي شكله L يحجب نفسه)، مكثّفة كل شوي"""
    idx = GBA_TREE.query(E.buffer(SEARCH + 5), predicate="intersects")
    idx = idx[~np.isin(idx, site_idx)]
    idx = idx[np.isfinite(GBA_H[idx]) & (GBA_H[idx] > 0)]
    geoms = np.concatenate([GBA_G[idx], np.array([E], dtype=object)]); hts = np.append(GBA_H[idx], Hb)
    # المسافة بين نقاط الجار لازم تكون أصغر من عرض القطاع 5° عند أقرب نقطة على جدارنا:
    # عرض 5° على بعد D = D·tan(5°) ≈ 0.087·D → ناخذ نصه تقريباً (0.04·D) ، و D = المسافة من الجار لجدار بيتنا
    D0 = np.maximum(shapely.distance(geoms, E.exterior), 0.5)
    step = np.clip(D0 * 0.04, 0.05, 4.0)
    xy, k = shapely.get_coordinates(shapely.segmentize(shapely.boundary(geoms), step), return_index=True)
    return xy[:, 0], xy[:, 1], hts[k], GBA_G[idx], GBA_H[idx]

def horizon(px, py, ph, ox, oy, oh, chunk=40):
    """β لكل نقطة × 72 قطاع: أكبر arctan((H − h) ÷ D) في القطاع، ضمن 100م؛ الأوطى من النقطة → 0
       (نحسب كل 40 نقطة مع بعض عشان ما تمتلي الذاكرة)"""
    beta = np.zeros((len(px), NSEC))
    for a in range(0, len(px), chunk):
        b_ = slice(a, a + chunk)
        dx = ox[None, :] - px[b_, None]; dy = oy[None, :] - py[b_, None]
        D = np.hypot(dx, dy); dH = oh[None, :] - ph[b_, None]
        ok = (D > 0.01) & (D <= SEARCH) & (dH > 0)
        r, c = np.nonzero(ok)
        np.maximum.at(beta, (r + a, sec_of(np.degrees(np.arctan2(dx[r, c], dy[r, c])) % 360)),
                      np.degrees(np.arctan2(dH[r, c], D[r, c])))
    return beta

# ======================= 4) كشف السماء للجدار =======================
def wall_svf(beta, nz):
    """Steger et al. 2022 Eq.10: SVF_p = Σ(w·t) / Σw  (0–1)
       w = max(0, cos(φ − n)) ، t = (π/2 − β − sinβ·cosβ) / (π/2)"""
    w = np.clip(np.cos(np.radians(SEC_C[None, :] - nz[:, None])), 0, None)
    b = np.radians(beta)
    t = (np.pi / 2 - b - np.sin(b) * np.cos(b)) / (np.pi / 2)
    return (w * t).sum(1) / w.sum(1)

# ======================= 5) الإشعاع =======================
def radiation(beta, nz, svf_raw, S):
    """Rad_p = Σh (DNI·max(0, cos(az−n)·cos(alt))·lit + DHI·SVF_raw) / 1000   [kWh/m² للفصل]
       open_p = نفس الشي بدون جيران: lit = 1 ، SVF_raw = 0.5"""
    up = S["alt"] > 0
    alt, az, dni, dhi = S["alt"][up], S["az"][up], S["dni"][up], S["dhi"][up]
    inc = np.clip(np.cos(np.radians(az[None, :] - nz[:, None])) * np.cos(np.radians(alt[None, :])), 0, None)
    lit = alt[None, :] > beta[:, sec_of(az)]                                # الشمس أعلى من الجار؟
    direct = dni[None, :] * inc
    rad  = ((direct * lit).sum(1) + dhi.sum() * svf_raw) / 1000
    opn  = (direct.sum(1) + dhi.sum() * 0.5) / 1000
    return rad, opn, lit.mean(1)

# ======================= 6) الحساب لكل صف + الحفظ المرحلي =======================
COLS = [f"Rad_summer_{x}" for x in DIRS] + [f"Rad_winter_{x}" for x in DIRS] + [f"Rad_block_{x}" for x in DIRS]
for c in COLS:
    if c not in d: d[c] = np.nan

def save():
    tmp = FILE + ".tmp"
    d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)

def house_of(P):
    """المبنية: شكلها من GBA / الفاضية: الغلاف النظامي"""
    E, site_idx = real_house(P)
    if E is not None: return E, site_idx
    P1, sides = parcel_sides(P)
    E, _ = envelope(P1, sides)
    return E, GBA_TREE.query(E, predicate="intersects")               # أي شي صغير داخل الغلاف ما نحسبه جار

def row_radiation(i):
    P = get_parcel(d.at[i, lat], d.at[i, lon])
    if P is None: return None
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    E, site_idx = house_of(P)
    Hb = float(d.at[i, "building_height"]) if pd.notna(d.at[i, "building_height"]) else VACANT_H
    pts = facade_points(E, Hb)
    ox, oy, oh, _, _ = obstacles(E, Hb, site_idx)
    B = horizon(pts.x.values, pts.y.values, pts.h.values, ox, oy, oh)
    nz = pts.nz.values
    svf_raw = wall_svf(B, nz) / 2                                        # 0–0.5 للجدار الرأسي
    rs, os_, _ = radiation(B, nz, svf_raw, SEA["Summer"])
    rw, _, _   = radiation(B, nz, svf_raw, SEA["Winter"])
    t = pd.DataFrame(dict(dir=wall_dir(nz, house_axis(E)), rs=rs, os=os_, rw=rw))
    g = t.groupby("dir"); out = {}
    for x, gg in g:
        out[f"Rad_summer_{x}"] = round(gg.rs.mean(), 2)                   # kWh/m² للفصل (متوسط النقاط)
        out[f"Rad_winter_{x}"] = round(gg.rw.mean(), 2)
        out[f"Rad_block_{x}"]  = round(1 - gg.rs.sum() / gg.os.sum(), 4)  # صيف
    return out

todo = d.index[d[COLS].isna().all(axis=1)]                               # يكمّل من وين وقف
print(f"باقي {len(todo)} من {len(d)} صف")
errors, t0 = 0, time.time()
for n, i in enumerate(todo, 1):
    try:
        out = row_radiation(i)
        if out:
            for k, v in out.items(): d.at[i, k] = v
        else: errors += 1
    except Exception as e:
        errors += 1; print(f"  صف {i}: {type(e).__name__}: {e}")
    if n % SAVE_EVERY == 0:
        save(); el = time.time() - t0
        print(f"  حُفظ {n}/{len(todo)} — باقي تقريباً {el / n * (len(todo) - n) / 60:.0f} دقيقة")
save()

print(f"\n✅ خلص — صفوف ما انحسبت: {errors}")
print("\nكم بيت فيه كل جهة:"); print(d[[f"Rad_summer_{x}" for x in DIRS]].notna().sum().to_string())
print("\nمعدل يومي صيف (kWh/m²/يوم):"); print((d[[f"Rad_summer_{x}" for x in DIRS]].mean() / 92).round(2).to_string())
print("\nمعدل يومي شتاء (kWh/m²/يوم):"); print((d[[f"Rad_winter_{x}" for x in DIRS]].mean() / 90).round(2).to_string())
print("\nRad_block صيف:"); print(d[[f"Rad_block_{x}" for x in DIRS]].describe().round(3).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
أعمدة الشوارع — العرض: STREETWIDTH | النوع: STREETCLASS
EPW: SAU_RI_Riyadh-Khalid.Intl.AP.404370_TMYx.2011-2025.epw
Summer: 2208 ساعة
Winter: 2160 ساعة
باقي 5791 من 5791 صف
  حُفظ 50/5791 — باقي تقريباً 10 دقيقة
  حُفظ 100/5791 — باقي تقريباً 12 دقيقة
  حُفظ 150/5791 — باقي تقريباً 12 دقيقة
  حُفظ 200/5791 — باقي تقريباً 13 دقيقة
  حُفظ 250/5791 — باقي تقريباً 12 دقيقة
  حُفظ 300/5791 — باقي تقريباً 12 دقيقة
  حُفظ 350/5791 — باقي تقريباً 12 دقيقة
  حُفظ 400/5791 — باقي تقريباً 11 دقيقة
  حُفظ 450/5791 — باقي تقريباً 11 دقيقة
  حُفظ 500/5791 — باقي تقريباً 11 دقيقة
  حُفظ 550/5791 — باقي تقريباً 11 دقيقة
  حُفظ 600/5791 — باقي تقريباً 11 دقيقة
  حُفظ 650/5791 — باقي تقريباً 10 دقيقة
  حُفظ 700/5791 — باقي تقريباً 10 دقيقة
  حُفظ 750/5791 — باقي تقريباً 11 دقيقة
  حُفظ 800/5791 — باقي تقريباً 10 دقيقة
  حُفظ 850/5791 — باقي تقريباً 10 دقيقة
  حُفظ 900/5791 — ب

In [ ]:
STUDENT = "deema"
import os, pandas as pd
FILE = f"/content/drive/MyDrive/Maamar/villas_{STUDENT}.csv"
d = pd.read_csv(FILE)
cols = [c for c in d.columns if c.startswith("Rad_summer_")]
bad = d[cols].isna().all(axis=1)
print(d.loc[bad, ["lat", "lon", "building_area", "building_height"]])   # نشوف الصف قبل الحذف
d = d[~bad]
tmp = FILE + ".tmp"; d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)
print(f"✅ انحذف {bad.sum()} صف — بقى {len(d)}")

            lat        lon  building_area  building_height
1755  24.592624  46.787149            0.0             14.0
✅ انحذف 1 صف — بقى 5790


# Rad Yard

In [ ]:
# ===================== ⚙️ الإعدادات: غيّري اسمك هنا بس =====================
STUDENT = "deema"          # deema | renad | jana | linah
TEST    = False             # True = تجربة على 3 قطع (رسم، بدون حفظ) | False = كل الملف مع الحفظ
SAVE_EVERY = 50            # يحفظ في ملفك كل كم صف
# ==========================================================================
# إشعاع الأرض (الحوش): Rad_yard صيف + شتاء لكل جهة من 8 — الزوايا منفصلة (16 عمود)

import os, glob, zipfile, time, numpy as np, pandas as pd, geopandas as gpd, shapely, pvlib
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point
from shapely.geometry.polygon import orient
from pyproj import Transformer
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Maamar"
FILE = f"{BASE}/villas_{STUDENT}.csv"

# ======================= الثوابت =======================
RES     = 2.0     # شبكة النقاط على أرض الحوش (م) — نفس شبكة الواجهات
NSEC    = 72      # 72 قطاع × 5° — shadow package (res_angle = 5)
SEARCH  = 100.0   # نبحث عن الجيران لين 100 م من النقطة
MIN_AREA = 20.0
DIRS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
SEASONS = {"Summer": ([6, 7, 8], 92), "Winter": ([12, 1, 2], 90)}
SEC_C = (np.arange(NSEC) + 0.5) * 360 / NSEC                     # منتصف كل قطاع
def sec_of(az): return (np.asarray(az) // (360 / NSEC)).astype(int) % NSEC
def dir8(az):   return DIRS[int(((az + 22.5) % 360) // 45)]
def adiff(a, b): return np.abs((a - b + 180) % 360 - 180)
TOL, MAX_ANGLE, CAND_RADIUS = 3.0, 30, 80          # الشارع الملاصق: حافته ≤3م وموازي ≤30°
NB_SETBACK, COVER_MAX = 1.5, 0.75                  # اشتراطات 4-1-4 و 4-1-1
VACANT_H = 14.0                                    # ارتفاع الفاضية (نفس كود الارتفاع)
def line_dir(line, pt):
    if line.geom_type == "MultiLineString":
        line = min(line.geoms, key=lambda g: g.distance(pt))
    s = line.project(pt)
    a, b = line.interpolate(max(s - 1, 0)), line.interpolate(min(s + 1, line.length))
    return np.degrees(np.arctan2(b.x - a.x, b.y - a.y)) % 180

def house_axis(E):
    """اتجاه البيت العام = أصغر مستطيل يحيط فيه (SMBR) — Duchêne et al. 2003"""
    R = np.asarray(shapely.minimum_rotated_rectangle(E).exterior.coords)
    (x0, y0), (x1, y1) = R[0], R[1]
    return np.degrees(np.arctan2(x1 - x0, y1 - y0)) % 90                 # محاور البيت تتكرر كل 90°

def wall_dir(nz, axis):
    """الجهة (8) لكل جدار بعد ما نربطه بمحاور البيت: محور البيت + مضاعفات 45°
       (الجدران المتوازية تاخذ نفس الجهة؛ الحساب الفيزيائي يبقى على زاوية الجدار الحقيقية)"""
    snapped = axis + np.round((np.asarray(nz) - axis) / 45) * 45
    return [dir8(a % 360) for a in snapped]

# ======================= البيانات =======================
d = pd.read_csv(FILE)
lat = next(c for c in d.columns if c.lower() in ("lat", "latitude"))
lon = next(c for c in d.columns if c.lower() in ("lon", "lng", "longitude"))
assert "building_height" in d, "شغّلي كود الارتفاع أول"

parcels = gpd.read_parquet(f"{BASE}/riyadh_parcels_all.parquet").to_crs(32638)
PA_G = shapely.make_valid(parcels.geometry.to_numpy()); PA_TREE = shapely.STRtree(PA_G)
PA_SUB = parcels["ParcelSub"].to_numpy(object) if "ParcelSub" in parcels else np.array([""] * len(parcels), object)

streets = gpd.read_parquet(f"{BASE}/riyadh_streets.parquet").to_crs(32638)
wcol = next((c for c in streets.columns if "width" in c.lower() or "عرض" in c), None)
ccol = next((c for c in streets.columns if c.upper() in ("SUBTYPECLASS", "CLASS", "STREETCLASS") or "تصنيف" in c), None)
print("أعمدة الشوارع — العرض:", wcol, "| النوع:", ccol)
streets["W"] = pd.to_numeric(streets[wcol], errors="coerce") if wcol else np.nan
streets["W"] = streets["W"].where(streets["W"] > 0)
if ccol: streets["W"] = streets["W"].fillna(streets.groupby(ccol)["W"].transform("median"))
streets["W"] = streets["W"].fillna(streets["W"].median())
ST_G = streets.geometry.to_numpy(); ST_W = streets["W"].to_numpy(float)
ST_C = streets[ccol].astype(str).to_numpy(object) if ccol else np.array([""] * len(streets), object)
ST_TREE = shapely.STRtree(ST_G)

gba = pd.concat([gpd.read_parquet(f"{BASE}/GBA/{t}.parquet")
                 for t in ["e045_n25_e050_n20", "e045_n30_e050_n25"]]).to_crs(32638)
hcol = next(c for c in gba.columns if "height" in c.lower())
GBA_G = gba.geometry.to_numpy(); GBA_TREE = shapely.STRtree(GBA_G)
GBA_H = pd.to_numeric(gba[hcol], errors="coerce").to_numpy(float)
TO_M = Transformer.from_crs(4326, 32638, always_xy=True)

# ---------- EPW: مطار الملك خالد TMYx ----------
epw_files = glob.glob(f"{BASE}/**/*.epw", recursive=True)
if not epw_files:
    z = glob.glob(f"{BASE}/**/*Riyadh*Khalid*.zip", recursive=True)[0]
    zipfile.ZipFile(z).extractall("/content/epw"); epw_files = glob.glob("/content/epw/**/*.epw", recursive=True)
EPW_FILE = next((f for f in epw_files if "Khalid" in f), epw_files[0]); print("EPW:", os.path.basename(EPW_FILE))
epw, meta = pvlib.iotools.read_epw(EPW_FILE)
# التوقيت: اختبار الإغلاق GHI = DNI·sin(alt) + DHI على نفس الملف → أقل خطأ عند الإزاحة 0
# (RMSE = 16.5 W/m²؛ +30 دقيقة = 42.7) → مكان الشمس عند epw.index نفسه
tm = epw.index
sp = pvlib.solarposition.get_solarposition(tm, meta["latitude"], meta["longitude"])   # NREL SPA
ALT, AZ = sp["apparent_elevation"].to_numpy(), sp["azimuth"].to_numpy()
DNI, DHI = epw["dni"].to_numpy(float), epw["dhi"].to_numpy(float)
SEA = {}
for s, (ms, days) in SEASONS.items():
    m = np.isin(tm.month, ms)
    SEA[s] = dict(alt=ALT[m], az=AZ[m], dni=DNI[m], dhi=DHI[m], days=days)
    print(f"{s}: {m.sum()} ساعة")

# ======================= 1) القطعة + البيت =======================
def get_parcel(la, lo):
    pt = Point(*TO_M.transform(lo, la))
    idx = PA_TREE.query(pt, predicate="intersects")
    if len(idx):
        res = np.array(["سكني" in str(PA_SUB[i]) for i in idx])
        if res.any(): idx = idx[res]
        return PA_G[idx[np.argmin(shapely.area(PA_G[idx]))]]
    idx = PA_TREE.query(pt.buffer(30), predicate="intersects")
    return PA_G[idx[np.argmin(shapely.distance(PA_G[idx], pt))]] if len(idx) else None

def real_house(P):
    """نفس كود الداتا سيت — ويرجّع أرقام مباني GBA اللي تكوّن البيت (عشان ما نحسبها جيران)"""
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    idx = GBA_TREE.query(P.buffer(5), predicate="intersects")
    if len(idx) == 0: return None, None
    g = GBA_G[idx]
    pc = P.centroid if P.contains(P.centroid) else P.representative_point()
    area = shapely.area(g); inter = shapely.area(shapely.intersection(g, P))
    frac = np.divide(inter, area, out=np.zeros_like(area), where=area > 0)
    site = shapely.within(shapely.centroid(g), P) | (frac >= 0.5) | shapely.contains(g, pc) \
           | (shapely.distance(g, pc) <= 2.0)
    if not site.any(): return None, None
    H = shapely.union_all(g[site]).intersection(P.buffer(1.0))
    if H.geom_type == "MultiPolygon": H = max(H.geoms, key=lambda q: q.area)
    return (H, idx[site]) if (not H.is_empty and H.area >= MIN_AREA) else (None, None)

# ======================= 1ب) أضلاع القطعة وشوارعها =======================
def parcel_sides(P):
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    ring = np.asarray(orient(P, 1.0).exterior.coords); sides = []
    for (x1, y1), (x2, y2) in zip(ring[:-1], ring[1:]):
        dx, dy = x2 - x1, y2 - y1; L = np.hypot(dx, dy)
        if L < 0.5: continue
        seg = LineString([(x1, y1), (x2, y2)]); mid = seg.interpolate(0.5, normalized=True)
        nz = np.degrees(np.arctan2(dy, -dx)) % 360; edir = np.degrees(np.arctan2(dx, dy)) % 180
        street = None
        idx = ST_TREE.query(mid.buffer(CAND_RADIUS))
        if len(idx):
            resid = shapely.distance(ST_G[idx], mid) - ST_W[idx] / 2
            for j in idx[np.argsort(np.abs(resid))]:
                if shapely.distance(ST_G[j], mid) - ST_W[j] / 2 > TOL: break
                par = adiff(line_dir(ST_G[j], mid), edir)
                if min(par, 180 - par) <= MAX_ANGLE: street = j; break
        sides.append(dict(seg=seg, nz=nz, dir=dir8(nz), length=L, street=street,
                          width=ST_W[street] if street is not None else np.nan,
                          cls=ST_C[street] if street is not None else None))
    return P, sides

# ======================= 1ج) الغلاف (للفاضية) =======================
def setback(width, cls, is_main):
    """اشتراطات 4-1-4 و 4-1-5 (صفحة 17)"""
    cls = cls if isinstance(cls, str) else ""
    w = width if pd.notna(width) and width > 0 else 0
    if "ممر" in cls:                  return 1.5
    if "ساح" in cls or "ميد" in cls:  return max(3.0, w / 5)
    if is_main:                        return 6.0 if w >= 30 else max(3.0, w / 5)
    return max(2.0, w / 5)

FALLBACK_R = 50.0                                   # أبعد مسافة نبحث فيها عن شارع قدّام الضلع (م)
def nearest_front_street(sides):
    """لو ما فيه شارع ملاصق: نختار الضلع اللي قدّامه أقرب شارع (حافة الشارع، مو خط المنتصف)"""
    best = None
    for s in sides:
        mid = s["seg"].interpolate(0.5, normalized=True)
        nx, ny = np.sin(np.radians(s["nz"])), np.cos(np.radians(s["nz"]))
        idx = ST_TREE.query(mid.buffer(FALLBACK_R))
        for j in idx:
            q = ST_G[j].interpolate(ST_G[j].project(mid))               # أقرب نقطة على الشارع
            vx, vy = q.x - mid.x, q.y - mid.y; dist = np.hypot(vx, vy)
            if dist > 0 and (vx * nx + vy * ny) / dist < np.cos(np.radians(60)):
                continue                                                # الشارع مو قدّام الضلع
            edge = max(dist - ST_W[j] / 2, 0)
            if edge <= FALLBACK_R and (best is None or edge < best[0]):
                best = (edge, s, j)
    if best is None: return False
    _, s, j = best
    s.update(street=j, width=ST_W[j], cls=ST_C[j])
    return True

def envelope(P, sides):
    st = [s for s in sides if s["street"] is not None]
    if not st and nearest_front_street(sides):                          # تصحيح: أقرب شارع قدّام القطعة
        st = [s for s in sides if s["street"] is not None]
    main = max(st, key=lambda s: (s["width"], s["length"])) if st else None
    cut, nb = [], []
    for s in sides:
        if s["street"] is not None:
            s["setback"] = setback(s["width"], s["cls"], s is main)
            s["type"] = "main street" if s is main else "street"
        else:
            s["setback"], s["type"] = NB_SETBACK, "neighbour"; nb.append(s["seg"])
        cut.append(s["seg"].buffer(s["setback"], cap_style=3))
    E = P.difference(shapely.union_all(cut))
    if E.geom_type == "MultiPolygon": E = max(E.geoms, key=lambda g: g.area)
    if E.is_empty or E.area < 0.2 * P.area: E = P
    if E.area > COVER_MAX * P.area and nb:                              # > 75%: نزيد من جهة الجيران
        nbu = shapely.union_all(nb); lo, hi = NB_SETBACK, NB_SETBACK + 10
        for _ in range(25):
            mid = (lo + hi) / 2
            Em = E.difference(nbu.buffer(mid, cap_style=3))
            Em = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
            lo, hi = (mid, hi) if Em.area > COVER_MAX * P.area else (lo, mid)
        Em = E.difference(nbu.buffer(hi, cap_style=3))
        E = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
        for s in sides:
            if s["type"] == "neighbour": s["setback"] = round(hi, 2)
    return E, main

# ======================= 2) نقاط الحوش + جهتها =======================
def yard_dir(gx, gy, E):
    """جهة كل نقطة في الحوش بالنسبة لأصغر مستطيل يحيط بالبيت (SMBR — Duchêne et al. 2003):
       قدّام ضلع المستطيل → جهة الضلع | برا زاويته → الزاوية (منفصلة) |
       داخل المستطيل (تجويف في البيت) → الضلع الأقرب نسبياً.
       كذا التجاويف في شكل البيت ما تطلّع زوايا وهمية في نص الحوش"""
    R = np.asarray(shapely.minimum_rotated_rectangle(E).exterior.coords)
    e1, e2 = R[1] - R[0], R[2] - R[1]
    a, b = np.linalg.norm(e1) / 2, np.linalg.norm(e2) / 2
    u1, u2 = e1 / (2 * a), e2 / (2 * b)
    c = R[:4].mean(0)
    du = (gx - c[0]) * u1[0] + (gy - c[1]) * u1[1]
    dv = (gx - c[0]) * u2[0] + (gy - c[1]) * u2[1]
    out_u, out_v = np.abs(du) > a, np.abs(dv) > b
    inside = ~out_u & ~out_v
    use_u = out_u | (inside & (np.abs(du) / a >= np.abs(dv) / b))
    use_v = out_v | (inside & ~use_u)
    vx = np.where(use_u, np.sign(du), 0)[:, None] * u1 + np.where(use_v, np.sign(dv), 0)[:, None] * u2
    az = np.degrees(np.arctan2(vx[:, 0], vx[:, 1])) % 360
    return wall_dir(az, house_axis(E))                                    # نفس محاور الواجهات

def yard_points(P, E):
    """شبكة 2×2 م على أرض الحوش (القطعة − البيت)، والنقطة في نص المربع"""
    Y = P.difference(E)
    if Y.is_empty: return pd.DataFrame(columns=["x", "y", "dir"]), Y
    x0, y0, x1, y1 = Y.bounds
    gx, gy = np.meshgrid(np.arange(x0 + RES / 2, x1, RES), np.arange(y0 + RES / 2, y1, RES))
    gx, gy = gx.ravel(), gy.ravel()
    inside = shapely.contains_xy(Y, gx, gy)
    gx, gy = gx[inside], gy[inside]
    if len(gx) == 0: return pd.DataFrame(columns=["x", "y", "dir"]), Y
    return pd.DataFrame(dict(x=gx, y=gy, dir=yard_dir(gx, gy, E))), Y

# ======================= 3) زاوية الأفق (h = 0) =======================
def obstacles_yard(E, Hb, site_idx, Y):
    """الجيران ضمن 100م + البيت نفسه (يحجب الحوش)، مكثّفة حسب بُعدها عن الحوش"""
    idx = GBA_TREE.query(Y.buffer(SEARCH + 5), predicate="intersects")
    idx = idx[~np.isin(idx, site_idx)]
    idx = idx[np.isfinite(GBA_H[idx]) & (GBA_H[idx] > 0)]
    geoms = np.concatenate([GBA_G[idx], np.array([E], dtype=object)]); hts = np.append(GBA_H[idx], Hb)
    D0 = np.maximum(shapely.distance(geoms, Y), 0.5)                     # أقرب مسافة لأي نقطة في الحوش
    step = np.clip(D0 * 0.04, 0.05, 4.0)                                 # أصغر من نص عرض القطاع 5°
    xy, k = shapely.get_coordinates(shapely.segmentize(shapely.boundary(geoms), step), return_index=True)
    return xy[:, 0], xy[:, 1], hts[k], GBA_G[idx]

def horizon(px, py, ph, ox, oy, oh, chunk=40):
    """β لكل نقطة × 72 قطاع: أكبر arctan((H − h) ÷ D) في القطاع، ضمن 100م؛ الأوطى من النقطة → 0
       (نحسب كل 40 نقطة مع بعض عشان ما تمتلي الذاكرة)"""
    beta = np.zeros((len(px), NSEC))
    for a in range(0, len(px), chunk):
        b_ = slice(a, a + chunk)
        dx = ox[None, :] - px[b_, None]; dy = oy[None, :] - py[b_, None]
        D = np.hypot(dx, dy); dH = oh[None, :] - ph[b_, None]
        ok = (D > 0.01) & (D <= SEARCH) & (dH > 0)
        r, c = np.nonzero(ok)
        np.maximum.at(beta, (r + a, sec_of(np.degrees(np.arctan2(dx[r, c], dy[r, c])) % 360)),
                      np.degrees(np.arctan2(dH[r, c], D[r, c])))
    return beta

# ======================= 4) كشف السماء للأرض + الإشعاع =======================
def ground_svf(beta):
    """Gal & Unger 2014, Eq.3: SVF = 1 − Σ sin²βᵢ / n   (0–1)"""
    return 1 - (np.sin(np.radians(beta)) ** 2).mean(1)

def ground_radiation(beta, svf, S):
    """Rad_Yard_p = Σh (DNI·sin(alt)·lit + DHI·SVF_أرض) / 1000   [kWh/m² للفصل]"""
    up = S["alt"] > 0
    alt, az, dni, dhi = S["alt"][up], S["az"][up], S["dni"][up], S["dhi"][up]
    lit = alt[None, :] > beta[:, sec_of(az)]
    return ((dni * np.sin(np.radians(alt)))[None, :] * lit).sum(1) / 1000 + dhi.sum() * svf / 1000

# ======================= 5) حساب صف واحد =======================
def house_of(P):
    E, site_idx = real_house(P)
    if E is not None: return E, site_idx
    P1, sides = parcel_sides(P)
    E, _ = envelope(P1, sides)
    return E, GBA_TREE.query(E, predicate="intersects")

def row_yard(i):
    P = get_parcel(d.at[i, lat], d.at[i, lon])
    if P is None: return None, None
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    E, site_idx = house_of(P)
    Hb = float(d.at[i, "building_height"]) if pd.notna(d.at[i, "building_height"]) else VACANT_H
    pts, Y = yard_points(P, E)
    if len(pts) == 0: return {}, (P, E, pts, [])
    ox, oy, oh, nb_g = obstacles_yard(E, Hb, site_idx, Y)
    B = horizon(pts.x.values, pts.y.values, np.zeros(len(pts)), ox, oy, oh)
    pts["svf"] = ground_svf(B)
    pts["summer"] = ground_radiation(B, pts.svf.values, SEA["Summer"])
    pts["winter"] = ground_radiation(B, pts.svf.values, SEA["Winter"])
    out = {}
    for x, g in pts.groupby("dir"):
        out[f"Rad_yard_summer_{x}"] = round(g.summer.mean(), 2)
        out[f"Rad_yard_winter_{x}"] = round(g.winter.mean(), 2)
    return out, (P, E, pts, nb_g)

COLS = [f"Rad_yard_summer_{x}" for x in DIRS] + [f"Rad_yard_winter_{x}" for x in DIRS]

# ======================= 6أ) تجربة =======================
if TEST:
    done = 0
    for i in d.sample(frac=1).index:
        if done == 3: break
        out, info = row_yard(i)
        if not out: continue
        done += 1; P, E, pts, nb_g = info
        print(f"\n========== القطعة {done}  ({d.at[i, lat]:.6f}, {d.at[i, lon]:.6f}) ==========")
        print(f"الأرض {P.area:.0f} م²  |  البيت {E.area:.0f} م²  |  الحوش {len(pts)} نقطة  |  الارتفاع {d.at[i, 'building_height']} م")
        g = pts.groupby("dir")
        print(pd.DataFrame({"عدد النقاط": g.size(), "SVF (0–1)": g.svf.mean().round(3),
                            "صيف kWh/m²": g.summer.mean().round(1), "صيف/يوم": (g.summer.mean() / 92).round(2),
                            "شتاء kWh/m²": g.winter.mean().round(1), "شتاء/يوم": (g.winter.mean() / 90).round(2)}
                           ).reindex([x for x in DIRS if x in g.groups]).to_string())
        print(f"https://www.google.com/maps?q={d.at[i, lat]},{d.at[i, lon]}&t=k")
        fig, axs = plt.subplots(1, 3, figsize=(17, 5.5))
        for ax, col, ttl in [(axs[0], None, "yard direction"), (axs[1], "summer", "summer kWh/m²"), (axs[2], "winter", "winter kWh/m²")]:
            gpd.GeoSeries(list(nb_g)).plot(ax=ax, color="#d0d0d0", edgecolor="#999", lw=.3)
            gpd.GeoSeries([P]).plot(ax=ax, facecolor="none", edgecolor="k", lw=1.2)
            gpd.GeoSeries([E]).plot(ax=ax, color="#f3efe6", edgecolor="#555")
            if col is None:
                for k_, (x, gg) in enumerate(pts.groupby("dir")):
                    ax.scatter(gg.x, gg.y, s=30, color=plt.cm.tab10(DIRS.index(x)), label=x, zorder=5)
                ax.legend(fontsize=8, loc="lower left")
            else:
                sc = ax.scatter(pts.x, pts.y, c=pts[col], cmap="inferno", s=30, zorder=5); plt.colorbar(sc, ax=ax, shrink=.7)
            x0, y0, x1, y1 = P.buffer(15).bounds; ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
            ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(ttl, fontsize=10)
        axs[0].annotate("N", xy=(.95, .95), xytext=(.95, .85), xycoords="axes fraction", ha="center",
                        arrowprops=dict(arrowstyle="->", lw=1.5))
        plt.show()

# ======================= 6ب) كل الملف + الحفظ المرحلي =======================
else:
    for c in COLS:
        if c not in d: d[c] = np.nan
    def save():
        tmp = FILE + ".tmp"
        d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)
    todo = d.index[d[COLS].isna().all(axis=1)]                           # يكمّل من وين وقف
    print(f"باقي {len(todo)} من {len(d)} صف")
    errors, no_yard, t0 = 0, 0, time.time()
    for n, i in enumerate(todo, 1):
        try:
            out, _ = row_yard(i)
            if out is None: errors += 1
            elif not out: no_yard += 1
            else:
                for k, v in out.items(): d.at[i, k] = v
        except Exception as e:
            errors += 1; print(f"  صف {i}: {type(e).__name__}: {e}")
        if n % SAVE_EVERY == 0:
            save(); el = time.time() - t0
            print(f"  حُفظ {n}/{len(todo)} — باقي تقريباً {el / n * (len(todo) - n) / 60:.0f} دقيقة")
    save()
    print(f"\n✅ خلص — بدون حوش: {no_yard} | أخطاء: {errors}")
    print("\nكم بيت فيه حوش في كل جهة:"); print(d[[f"Rad_yard_summer_{x}" for x in DIRS]].notna().sum().to_string())
    print("\nمعدل يومي صيف (kWh/m²/يوم):"); print((d[[f"Rad_yard_summer_{x}" for x in DIRS]].mean() / 92).round(2).to_string())
    print("\nمعدل يومي شتاء (kWh/m²/يوم):"); print((d[[f"Rad_yard_winter_{x}" for x in DIRS]].mean() / 90).round(2).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
أعمدة الشوارع — العرض: STREETWIDTH | النوع: STREETCLASS
EPW: SAU_RI_Riyadh-Khalid.Intl.AP.404370_TMYx.2011-2025.epw
Summer: 2208 ساعة
Winter: 2160 ساعة
باقي 5790 من 5790 صف
  حُفظ 50/5790 — باقي تقريباً 7 دقيقة
  حُفظ 100/5790 — باقي تقريباً 6 دقيقة
  حُفظ 150/5790 — باقي تقريباً 5 دقيقة
  حُفظ 200/5790 — باقي تقريباً 5 دقيقة
  حُفظ 250/5790 — باقي تقريباً 5 دقيقة
  حُفظ 300/5790 — باقي تقريباً 5 دقيقة
  حُفظ 350/5790 — باقي تقريباً 5 دقيقة
  حُفظ 400/5790 — باقي تقريباً 5 دقيقة
  حُفظ 450/5790 — باقي تقريباً 5 دقيقة
  حُفظ 500/5790 — باقي تقريباً 5 دقيقة
  حُفظ 550/5790 — باقي تقريباً 5 دقيقة
  حُفظ 600/5790 — باقي تقريباً 5 دقيقة
  حُفظ 650/5790 — باقي تقريباً 5 دقيقة
  حُفظ 700/5790 — باقي تقريباً 5 دقيقة
  حُفظ 750/5790 — باقي تقريباً 5 دقيقة
  حُفظ 800/5790 — باقي تقريباً 5 دقيقة
  حُفظ 850/5790 — باقي تقريباً 5 دقيقة
  حُفظ 900/5790 — باقي تقريباً 5 دقي

# Wind block

In [ ]:
# ===================== ⚙️ الإعدادات: غيّري اسمك هنا بس =====================
STUDENT = "deema"          # deema | renad | jana | linah
TEST    = False             # True = تجربة على 3 قطع (رسم، بدون حفظ) | False = كل الملف مع الحفظ
SAVE_EVERY = 50            # يحفظ في ملفك كل كم صف
# ==========================================================================
# حجب الرياح لكل واجهة: Wind_block_good (المرغوبة) + Wind_block_bad (الحارة/الباردة) — 16 عمود، بالدرجات (0–90)
# وزن الرياح حسب زاويتها على الجدار: Swami & Chandra 1987

import os, glob, zipfile, time, numpy as np, pandas as pd, geopandas as gpd, shapely, pvlib
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point
from shapely.geometry.polygon import orient
from pyproj import Transformer
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Maamar"
FILE = f"{BASE}/villas_{STUDENT}.csv"

# ======================= الثوابت =======================
RES     = 2.0     # شبكة النقاط على الجدار (م) — Chatzipoulka et al. 2018
NSEC    = 72      # 72 قطاع × 5° — shadow package (res_angle = 5)
SEARCH  = 100.0   # نبحث عن الجيران لين 100 م من النقطة
OFFSET  = 0.10    # النقطة برا الجدار بـ10 سم عشان الجدار ما يحجب نفسه
MIN_AREA = 20.0
DIRS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
SEC_C = (np.arange(NSEC) + 0.5) * 360 / NSEC                     # منتصف كل قطاع
def sec_of(az): return (np.asarray(az) // (360 / NSEC)).astype(int) % NSEC
def dir8(az):   return DIRS[int(((az + 22.5) % 360) // 45)]
def adiff(a, b): return np.abs((a - b + 180) % 360 - 180)
TOL, MAX_ANGLE, CAND_RADIUS = 3.0, 30, 80          # الشارع الملاصق: حافته ≤3م وموازي ≤30°
NB_SETBACK, COVER_MAX = 1.5, 0.75                  # اشتراطات 4-1-4 و 4-1-1
VACANT_H = 14.0                                    # ارتفاع الفاضية (نفس كود الارتفاع)
def line_dir(line, pt):
    if line.geom_type == "MultiLineString":
        line = min(line.geoms, key=lambda g: g.distance(pt))
    s = line.project(pt)
    a, b = line.interpolate(max(s - 1, 0)), line.interpolate(min(s + 1, line.length))
    return np.degrees(np.arctan2(b.x - a.x, b.y - a.y)) % 180

def house_axis(E):
    """اتجاه البيت العام = أصغر مستطيل يحيط فيه (SMBR) — Duchêne et al. 2003"""
    R = np.asarray(shapely.minimum_rotated_rectangle(E).exterior.coords)
    (x0, y0), (x1, y1) = R[0], R[1]
    return np.degrees(np.arctan2(x1 - x0, y1 - y0)) % 90                 # محاور البيت تتكرر كل 90°

def wall_dir(nz, axis):
    """الجهة (8) لكل جدار بعد ما نربطه بمحاور البيت: محور البيت + مضاعفات 45°
       (الجدران المتوازية تاخذ نفس الجهة؛ الحساب الفيزيائي يبقى على زاوية الجدار الحقيقية)"""
    snapped = axis + np.round((np.asarray(nz) - axis) / 45) * 45
    return [dir8(a % 360) for a in snapped]

# ======================= البيانات =======================
d = pd.read_csv(FILE)
lat = next(c for c in d.columns if c.lower() in ("lat", "latitude"))
lon = next(c for c in d.columns if c.lower() in ("lon", "lng", "longitude"))
assert "building_height" in d, "شغّلي كود الارتفاع أول"

parcels = gpd.read_parquet(f"{BASE}/riyadh_parcels_all.parquet").to_crs(32638)
PA_G = shapely.make_valid(parcels.geometry.to_numpy()); PA_TREE = shapely.STRtree(PA_G)
PA_SUB = parcels["ParcelSub"].to_numpy(object) if "ParcelSub" in parcels else np.array([""] * len(parcels), object)

streets = gpd.read_parquet(f"{BASE}/riyadh_streets.parquet").to_crs(32638)
wcol = next((c for c in streets.columns if "width" in c.lower() or "عرض" in c), None)
ccol = next((c for c in streets.columns if c.upper() in ("SUBTYPECLASS", "CLASS", "STREETCLASS") or "تصنيف" in c), None)
print("أعمدة الشوارع — العرض:", wcol, "| النوع:", ccol)
streets["W"] = pd.to_numeric(streets[wcol], errors="coerce") if wcol else np.nan
streets["W"] = streets["W"].where(streets["W"] > 0)
if ccol: streets["W"] = streets["W"].fillna(streets.groupby(ccol)["W"].transform("median"))
streets["W"] = streets["W"].fillna(streets["W"].median())
ST_G = streets.geometry.to_numpy(); ST_W = streets["W"].to_numpy(float)
ST_C = streets[ccol].astype(str).to_numpy(object) if ccol else np.array([""] * len(streets), object)
ST_TREE = shapely.STRtree(ST_G)

gba = pd.concat([gpd.read_parquet(f"{BASE}/GBA/{t}.parquet")
                 for t in ["e045_n25_e050_n20", "e045_n30_e050_n25"]]).to_crs(32638)
hcol = next(c for c in gba.columns if "height" in c.lower())
GBA_G = gba.geometry.to_numpy(); GBA_TREE = shapely.STRtree(GBA_G)
GBA_H = pd.to_numeric(gba[hcol], errors="coerce").to_numpy(float)
TO_M = Transformer.from_crs(4326, 32638, always_xy=True)

# ---------- EPW: مطار الملك خالد TMYx ----------
epw_files = glob.glob(f"{BASE}/**/*.epw", recursive=True)
if not epw_files:
    z = glob.glob(f"{BASE}/**/*Riyadh*Khalid*.zip", recursive=True)[0]
    zipfile.ZipFile(z).extractall("/content/epw"); epw_files = glob.glob("/content/epw/**/*.epw", recursive=True)
EPW_FILE = next((f for f in epw_files if "Khalid" in f), epw_files[0]); print("EPW:", os.path.basename(EPW_FILE))
epw, meta = pvlib.iotools.read_epw(EPW_FILE)

# ======================= الرياح: تصنيف كل ساعة (ASHRAE 55 التكيّفي) =======================
T, WS = epw["temp_air"].to_numpy(float), epw["wind_speed"].to_numpy(float)
WD = epw["wind_direction"].to_numpy(float) % 360                        # 360 = 0 (الاتجاه اللي تجي منه)
MON = epw.index.month
Tm = pd.Series(T).groupby(MON).transform("mean").to_numpy()             # متوسط كل ساعات الشهر
LO, HI = 0.31 * Tm + 14.3, 0.31 * Tm + 21.3                             # حد القبول 80% — de Dear & Brager 2002
APPLY = (Tm >= 10) & (Tm <= 33.5)                                       # حدود النموذج (ASHRAE 55)
WINDY = WS > 0                                                          # الساعات الهادية ما لها اتجاه
GOOD = APPLY & (T >= LO) & (T <= HI) & WINDY
BAD  = ~GOOD & WINDY
K = sec_of(WD)
F = {"good": np.bincount(K[GOOD], minlength=NSEC) / len(T),             # F(k) = عدد الساعات ÷ 8760
     "bad":  np.bincount(K[BAD],  minlength=NSEC) / len(T)}
print(f"مرغوبة: {GOOD.sum()} ساعة | غير مرغوبة: {BAD.sum()} | بدون رياح: {(~WINDY).sum()}  (المجموع {len(T)})")

# ======================= 1) القطعة + البيت =======================
def get_parcel(la, lo):
    pt = Point(*TO_M.transform(lo, la))
    idx = PA_TREE.query(pt, predicate="intersects")
    if len(idx):
        res = np.array(["سكني" in str(PA_SUB[i]) for i in idx])
        if res.any(): idx = idx[res]
        return PA_G[idx[np.argmin(shapely.area(PA_G[idx]))]]
    idx = PA_TREE.query(pt.buffer(30), predicate="intersects")
    return PA_G[idx[np.argmin(shapely.distance(PA_G[idx], pt))]] if len(idx) else None

def real_house(P):
    """نفس كود الداتا سيت — ويرجّع أرقام مباني GBA اللي تكوّن البيت (عشان ما نحسبها جيران)"""
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    idx = GBA_TREE.query(P.buffer(5), predicate="intersects")
    if len(idx) == 0: return None, None
    g = GBA_G[idx]
    pc = P.centroid if P.contains(P.centroid) else P.representative_point()
    area = shapely.area(g); inter = shapely.area(shapely.intersection(g, P))
    frac = np.divide(inter, area, out=np.zeros_like(area), where=area > 0)
    site = shapely.within(shapely.centroid(g), P) | (frac >= 0.5) | shapely.contains(g, pc) \
           | (shapely.distance(g, pc) <= 2.0)
    if not site.any(): return None, None
    H = shapely.union_all(g[site]).intersection(P.buffer(1.0))
    if H.geom_type == "MultiPolygon": H = max(H.geoms, key=lambda q: q.area)
    return (H, idx[site]) if (not H.is_empty and H.area >= MIN_AREA) else (None, None)

# ======================= 1ب) أضلاع القطعة وشوارعها =======================
def parcel_sides(P):
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    ring = np.asarray(orient(P, 1.0).exterior.coords); sides = []
    for (x1, y1), (x2, y2) in zip(ring[:-1], ring[1:]):
        dx, dy = x2 - x1, y2 - y1; L = np.hypot(dx, dy)
        if L < 0.5: continue
        seg = LineString([(x1, y1), (x2, y2)]); mid = seg.interpolate(0.5, normalized=True)
        nz = np.degrees(np.arctan2(dy, -dx)) % 360; edir = np.degrees(np.arctan2(dx, dy)) % 180
        street = None
        idx = ST_TREE.query(mid.buffer(CAND_RADIUS))
        if len(idx):
            resid = shapely.distance(ST_G[idx], mid) - ST_W[idx] / 2
            for j in idx[np.argsort(np.abs(resid))]:
                if shapely.distance(ST_G[j], mid) - ST_W[j] / 2 > TOL: break
                par = adiff(line_dir(ST_G[j], mid), edir)
                if min(par, 180 - par) <= MAX_ANGLE: street = j; break
        sides.append(dict(seg=seg, nz=nz, dir=dir8(nz), length=L, street=street,
                          width=ST_W[street] if street is not None else np.nan,
                          cls=ST_C[street] if street is not None else None))
    return P, sides

# ======================= 1ج) الغلاف (للفاضية) =======================
def setback(width, cls, is_main):
    """اشتراطات 4-1-4 و 4-1-5 (صفحة 17)"""
    cls = cls if isinstance(cls, str) else ""
    w = width if pd.notna(width) and width > 0 else 0
    if "ممر" in cls:                  return 1.5
    if "ساح" in cls or "ميد" in cls:  return max(3.0, w / 5)
    if is_main:                        return 6.0 if w >= 30 else max(3.0, w / 5)
    return max(2.0, w / 5)

FALLBACK_R = 50.0                                   # أبعد مسافة نبحث فيها عن شارع قدّام الضلع (م)
def nearest_front_street(sides):
    """لو ما فيه شارع ملاصق: نختار الضلع اللي قدّامه أقرب شارع (حافة الشارع، مو خط المنتصف)"""
    best = None
    for s in sides:
        mid = s["seg"].interpolate(0.5, normalized=True)
        nx, ny = np.sin(np.radians(s["nz"])), np.cos(np.radians(s["nz"]))
        idx = ST_TREE.query(mid.buffer(FALLBACK_R))
        for j in idx:
            q = ST_G[j].interpolate(ST_G[j].project(mid))               # أقرب نقطة على الشارع
            vx, vy = q.x - mid.x, q.y - mid.y; dist = np.hypot(vx, vy)
            if dist > 0 and (vx * nx + vy * ny) / dist < np.cos(np.radians(60)):
                continue                                                # الشارع مو قدّام الضلع
            edge = max(dist - ST_W[j] / 2, 0)
            if edge <= FALLBACK_R and (best is None or edge < best[0]):
                best = (edge, s, j)
    if best is None: return False
    _, s, j = best
    s.update(street=j, width=ST_W[j], cls=ST_C[j])
    return True

def envelope(P, sides):
    st = [s for s in sides if s["street"] is not None]
    if not st and nearest_front_street(sides):                          # تصحيح: أقرب شارع قدّام القطعة
        st = [s for s in sides if s["street"] is not None]
    main = max(st, key=lambda s: (s["width"], s["length"])) if st else None
    cut, nb = [], []
    for s in sides:
        if s["street"] is not None:
            s["setback"] = setback(s["width"], s["cls"], s is main)
            s["type"] = "main street" if s is main else "street"
        else:
            s["setback"], s["type"] = NB_SETBACK, "neighbour"; nb.append(s["seg"])
        cut.append(s["seg"].buffer(s["setback"], cap_style=3))
    E = P.difference(shapely.union_all(cut))
    if E.geom_type == "MultiPolygon": E = max(E.geoms, key=lambda g: g.area)
    if E.is_empty or E.area < 0.2 * P.area: E = P
    if E.area > COVER_MAX * P.area and nb:                              # > 75%: نزيد من جهة الجيران
        nbu = shapely.union_all(nb); lo, hi = NB_SETBACK, NB_SETBACK + 10
        for _ in range(25):
            mid = (lo + hi) / 2
            Em = E.difference(nbu.buffer(mid, cap_style=3))
            Em = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
            lo, hi = (mid, hi) if Em.area > COVER_MAX * P.area else (lo, mid)
        Em = E.difference(nbu.buffer(hi, cap_style=3))
        E = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
        for s in sides:
            if s["type"] == "neighbour": s["setback"] = round(hi, 2)
    return E, main

# ======================= 2) نقاط الواجهات =======================
def facade_points(E, Hb):
    """كل جدار: مربعات 2×2 م والنقطة في نص المربع، برا الجدار بـ10 سم"""
    hs = np.arange(RES / 2, Hb, RES) if Hb > RES else np.array([Hb / 2])
    ring = np.asarray(orient(E, 1.0).exterior.coords)
    X, Y, Z, NZ, S, W = [], [], [], [], [], []; s0 = 0.0
    for k, ((x1, y1), (x2, y2)) in enumerate(zip(ring[:-1], ring[1:])):
        dx, dy = x2 - x1, y2 - y1; L = np.hypot(dx, dy)
        if L < 1.0: s0 += L; continue
        nx, ny = dy / L, -dx / L                                          # العمود للخارج
        n = max(1, int(round(L / RES))); t = (np.arange(n) + 0.5) / n
        px, py = x1 + t * dx + nx * OFFSET, y1 + t * dy + ny * OFFSET
        nz = np.degrees(np.arctan2(nx, ny)) % 360
        for h in hs:
            X.append(px); Y.append(py); Z.append(np.full(n, h)); NZ.append(np.full(n, nz))
            S.append(s0 + t * L); W.append(np.full(n, k))
        s0 += L
    c = np.concatenate
    return pd.DataFrame(dict(x=c(X), y=c(Y), h=c(Z), nz=c(NZ), s=c(S), wall=c(W)))

# ======================= 3) زاوية الأفق =======================
def obstacles(E, Hb, site_idx):
    """حدود الجيران (ضمن 100م) + بيتنا نفسه (عشان البيت اللي شكله L يحجب نفسه)، مكثّفة كل شوي"""
    idx = GBA_TREE.query(E.buffer(SEARCH + 5), predicate="intersects")
    idx = idx[~np.isin(idx, site_idx)]
    idx = idx[np.isfinite(GBA_H[idx]) & (GBA_H[idx] > 0)]
    geoms = np.concatenate([GBA_G[idx], np.array([E], dtype=object)]); hts = np.append(GBA_H[idx], Hb)
    # المسافة بين نقاط الجار لازم تكون أصغر من عرض القطاع 5° عند أقرب نقطة على جدارنا:
    # عرض 5° على بعد D = D·tan(5°) ≈ 0.087·D → ناخذ نصه تقريباً (0.04·D) ، و D = المسافة من الجار لجدار بيتنا
    D0 = np.maximum(shapely.distance(geoms, E.exterior), 0.5)
    step = np.clip(D0 * 0.04, 0.05, 4.0)
    xy, k = shapely.get_coordinates(shapely.segmentize(shapely.boundary(geoms), step), return_index=True)
    return xy[:, 0], xy[:, 1], hts[k], GBA_G[idx], GBA_H[idx]

def horizon(px, py, ph, ox, oy, oh, chunk=40):
    """β لكل نقطة × 72 قطاع: أكبر arctan((H − h) ÷ D) في القطاع، ضمن 100م؛ الأوطى من النقطة → 0
       (نحسب كل 40 نقطة مع بعض عشان ما تمتلي الذاكرة)"""
    beta = np.zeros((len(px), NSEC))
    for a in range(0, len(px), chunk):
        b_ = slice(a, a + chunk)
        dx = ox[None, :] - px[b_, None]; dy = oy[None, :] - py[b_, None]
        D = np.hypot(dx, dy); dH = oh[None, :] - ph[b_, None]
        ok = (D > 0.01) & (D <= SEARCH) & (dH > 0)
        r, c = np.nonzero(ok)
        np.maximum.at(beta, (r + a, sec_of(np.degrees(np.arctan2(dx[r, c], dy[r, c])) % 360)),
                      np.degrees(np.arctan2(dH[r, c], D[r, c])))
    return beta

# ======================= 4) حجب الرياح =======================
def cp_ratio(alpha, G):
    """ضغط الرياح على الجدار نسبةً للرياح المباشرة — Swami & Chandra 1987 (FSEC-CR-163-86)
       alpha = زاوية الرياح على الجدار (0 = مباشرة) ، G = ln(طول الجدار ÷ طول الجدار المجاور)"""
    a = np.radians(alpha)
    arg = (1.248 - 0.703 * np.sin(a / 2) - 1.175 * np.sin(a) ** 2 - 0.131 * np.sin(2 * a * G) ** 3
           + 0.769 * np.cos(a / 2) + 0.071 * G ** 2 * np.sin(a / 2) ** 2 + 0.717 * np.cos(a / 2) ** 2)
    return np.log(np.clip(arg, 1e-9, None))

def wall_G(nz, E):
    """G لكل نقطة من أصغر مستطيل يحيط بالبيت: الجدار الموازي للضلع الطويل ياخذ ln(طويل/قصير) والعكس"""
    R = np.asarray(shapely.minimum_rotated_rectangle(E).exterior.coords)
    e1, e2 = R[1] - R[0], R[2] - R[1]
    L1, L2 = np.linalg.norm(e1), np.linalg.norm(e2)
    az1 = np.degrees(np.arctan2(e1[0], e1[1])) % 180                     # اتجاه الضلع الأول
    dd = adiff(nz, az1) % 180; dd = np.minimum(dd, 180 - dd)
    normal_along_e1 = dd < 45                                            # الجدار عمودي على e1 → طوله L2
    return np.where(normal_along_e1, np.log(L2 / L1), np.log(L1 / L2))

def wind_block(B, nz, G, Fk):
    """Wind_block_p = Σk F(k)·w(k)·β_p(k) / Σk F(k)·w(k)
       w(k) = max(0, Cp/Cp0) — Swami & Chandra 1987: المباشرة وزنها 1، تقل مع الميلان،
       والرياح اللي تسبب شفط (تقريباً بعد 63–72°) وزنها 0.  لو Σ = 0 → NaN"""
    alpha = adiff(SEC_C[None, :], nz[:, None])
    w = np.clip(cp_ratio(alpha, G[:, None]), 0, None)
    W = Fk[None, :] * w
    den = W.sum(1)
    return np.divide((W * B).sum(1), den, out=np.full(len(nz), np.nan), where=den > 0)

# ======================= 5) حساب صف واحد =======================
def house_of(P):
    E, site_idx = real_house(P)
    if E is not None: return E, site_idx
    P1, sides = parcel_sides(P)
    E, _ = envelope(P1, sides)
    return E, GBA_TREE.query(E, predicate="intersects")

def row_wind(i):
    P = get_parcel(d.at[i, lat], d.at[i, lon])
    if P is None: return None, None
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    E, site_idx = house_of(P)
    Hb = float(d.at[i, "building_height"]) if pd.notna(d.at[i, "building_height"]) else VACANT_H
    pts = facade_points(E, Hb)
    ox, oy, oh, nb_g, _ = obstacles(E, Hb, site_idx)
    B = horizon(pts.x.values, pts.y.values, pts.h.values, ox, oy, oh)
    nz = pts.nz.values
    pts["dir"] = wall_dir(nz, house_axis(E))
    G = wall_G(nz, E)
    for t in ("good", "bad"): pts[t] = wind_block(B, nz, G, F[t])
    out = {}
    for x, g in pts.groupby("dir"):
        for t in ("good", "bad"):
            v = g[t].mean()                                              # متوسط النقاط (NaN لو ما فيه رياح)
            out[f"Wind_block_{t}_{x}"] = round(v, 2) if pd.notna(v) else np.nan
    return out, (P, E, pts, nb_g)

COLS = [f"Wind_block_good_{x}" for x in DIRS] + [f"Wind_block_bad_{x}" for x in DIRS]

# ======================= 6أ) تجربة =======================
if TEST:
    # --- تصنيف الساعات لكل شهر ---
    m = pd.DataFrame({"T_m": pd.Series(T).groupby(MON).mean().round(1),
                      "lo": pd.Series(LO).groupby(MON).first().round(1),
                      "hi": pd.Series(HI).groupby(MON).first().round(1),
                      "النموذج ينطبق": pd.Series(APPLY).groupby(MON).first(),
                      "مرغوبة": pd.Series(GOOD).groupby(MON).sum(),
                      "غير مرغوبة": pd.Series(BAD).groupby(MON).sum(),
                      "بدون رياح": pd.Series(~WINDY).groupby(MON).sum()})
    m.index.name = "الشهر"; print(m.to_string())

    # --- وردة الرياح (المرغوبة وغير المرغوبة) ---
    fig = plt.figure(figsize=(11, 5))
    for j, t in enumerate(("good", "bad")):
        ax = fig.add_subplot(1, 2, j + 1, projection="polar")
        ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
        ax.bar(np.radians(SEC_C), F[t] * 100, width=np.radians(5), color="#2ca02c" if t == "good" else "#d62728", alpha=.8)
        ax.set_title(f"wind rose — {t} hours (% of year, from)", fontsize=10)
    plt.show()

    done = 0
    for i in d.sample(frac=1).index:
        if done == 3: break
        out, info = row_wind(i)
        if not out: continue
        done += 1; P, E, pts, nb_g = info
        g = pts.groupby("dir")
        print(f"\n========== القطعة {done}  ({d.at[i, lat]:.6f}, {d.at[i, lon]:.6f}) ==========")
        print(f"البيت {E.area:.0f} م²  |  الارتفاع {d.at[i, 'building_height']} م  |  {len(pts)} نقطة")
        print(pd.DataFrame({"عدد النقاط": g.size(),
                            "Wind_block_good °": g.good.mean().round(2),
                            "Wind_block_bad °": g.bad.mean().round(2)}
                           ).reindex([x for x in DIRS if x in g.groups]).to_string())
        print(f"https://www.google.com/maps?q={d.at[i, lat]},{d.at[i, lon]}&t=k")
        fig, axs = plt.subplots(1, 2, figsize=(12, 5.5))
        col = pts.groupby(["x", "y"], as_index=False)[["good", "bad"]].mean()
        for ax, t in zip(axs, ("good", "bad")):
            gpd.GeoSeries(list(nb_g)).plot(ax=ax, color="#d0d0d0", edgecolor="#999", lw=.3)
            gpd.GeoSeries([P]).plot(ax=ax, facecolor="none", edgecolor="k", lw=1.2)
            gpd.GeoSeries([E]).plot(ax=ax, color="#f3efe6", edgecolor="#555")
            sc = ax.scatter(col.x, col.y, c=col[t], cmap="viridis_r", s=22, zorder=5, vmin=0)
            plt.colorbar(sc, ax=ax, shrink=.7, label="blocking angle ° (mean of column)")
            x0, y0, x1, y1 = P.buffer(25).bounds; ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
            ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"Wind_block_{t}", fontsize=10)
        axs[0].annotate("N", xy=(.95, .95), xytext=(.95, .85), xycoords="axes fraction", ha="center",
                        arrowprops=dict(arrowstyle="->", lw=1.5))
        plt.show()

# ======================= 6ب) كل الملف + الحفظ المرحلي =======================
else:
    for c in COLS:
        if c not in d: d[c] = np.nan
    def save():
        tmp = FILE + ".tmp"
        d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)
    todo = d.index[d[COLS].isna().all(axis=1)]                           # يكمّل من وين وقف
    print(f"باقي {len(todo)} من {len(d)} صف")
    errors, t0 = 0, time.time()
    for n, i in enumerate(todo, 1):
        try:
            out, _ = row_wind(i)
            if not out: errors += 1
            else:
                for k, v in out.items(): d.at[i, k] = v
        except Exception as e:
            errors += 1; print(f"  صف {i}: {type(e).__name__}: {e}")
        if n % SAVE_EVERY == 0:
            save(); el = time.time() - t0
            print(f"  حُفظ {n}/{len(todo)} — باقي تقريباً {el / n * (len(todo) - n) / 60:.0f} دقيقة")
    save()
    print(f"\n✅ خلص — صفوف ما انحسبت: {errors}")
    print("\nWind_block_good (°):"); print(d[[f"Wind_block_good_{x}" for x in DIRS]].describe().round(2).to_string())
    print("\nWind_block_bad (°):");  print(d[[f"Wind_block_bad_{x}" for x in DIRS]].describe().round(2).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
أعمدة الشوارع — العرض: STREETWIDTH | النوع: STREETCLASS
EPW: SAU_RI_Riyadh-Khalid.Intl.AP.404370_TMYx.2011-2025.epw
مرغوبة: 1888 ساعة | غير مرغوبة: 5666 | بدون رياح: 1206  (المجموع 8760)
باقي 5790 من 5790 صف
  حُفظ 50/5790 — باقي تقريباً 9 دقيقة
  حُفظ 100/5790 — باقي تقريباً 10 دقيقة
  حُفظ 150/5790 — باقي تقريباً 10 دقيقة
  حُفظ 200/5790 — باقي تقريباً 10 دقيقة
  حُفظ 250/5790 — باقي تقريباً 10 دقيقة
  حُفظ 300/5790 — باقي تقريباً 10 دقيقة
  حُفظ 350/5790 — باقي تقريباً 10 دقيقة
  حُفظ 400/5790 — باقي تقريباً 10 دقيقة
  حُفظ 450/5790 — باقي تقريباً 10 دقيقة
  حُفظ 500/5790 — باقي تقريباً 10 دقيقة
  حُفظ 550/5790 — باقي تقريباً 10 دقيقة
  حُفظ 600/5790 — باقي تقريباً 10 دقيقة
  حُفظ 650/5790 — باقي تقريباً 9 دقيقة
  حُفظ 700/5790 — باقي تقريباً 9 دقيقة
  حُفظ 750/5790 — باقي تقريباً 10 دقيقة
  حُفظ 800/5790 — باقي تقريباً 9 دقيقة
  حُفظ 850/5790 — باقي تقريب

# Noise and facade length

In [ ]:
# ===================== ⚙️ الإعدادات: غيّري اسمك هنا بس =====================
STUDENT = "deema"          # deema | renad | jana | linah
TEST    = False             # True = تجربة على 3 قطع (رسم، بدون حفظ) | False = كل الملف مع الحفظ
SAVE_EVERY = 100           # يحفظ في ملفك كل كم صف
# ==========================================================================
# الضوضاء + أعمدة B (الخام) — 75 عمود:
#   لكل جهة (× 8): Noise ، dist_major ، dist_shop ، facade_length
#   للقطعة: dist_road_{نوع} (6) ، len_{نوع}_{R}m (31) ، n_bldg_{R}m (6)

import os, json, re, time, numpy as np, pandas as pd, geopandas as gpd, shapely
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point, Polygon
from shapely.geometry.polygon import orient
from pyproj import Transformer
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Maamar"
FILE = f"{BASE}/villas_{STUDENT}.csv"

# ======================= الثوابت =======================
DIRS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
def dir8(az):   return DIRS[int(((az + 22.5) % 360) // 45)]
def adiff(a, b): return np.abs((a - b + 180) % 360 - 180)
def line_dir(line, pt):
    if line.geom_type == "MultiLineString":
        line = min(line.geoms, key=lambda g: g.distance(pt))
    s = line.project(pt)
    a, b = line.interpolate(max(s - 1, 0)), line.interpolate(min(s + 1, line.length))
    return np.degrees(np.arctan2(b.x - a.x, b.y - a.y)) % 180
MIN_AREA = 20.0
VACANT_H = 14.0
TOL, MAX_ANGLE, CAND_RADIUS = 3.0, 30, 80          # الشارع الملاصق: حافته ≤3م وموازي ≤30°
NB_SETBACK, COVER_MAX = 1.5, 0.75                  # اشتراطات 4-1-4 و 4-1-1

# ---- قانون الضوضاء: R (الشوارع) — حدود إرشادات أوتاوا ----
R_LIMIT = {**{c: 500 for c in ["طريق سريع", "مخرج", "مدخل"]},                                   # motorway
           **{c: 100 for c in ["طريق رئيسي", "كوبري",                                           # arterial
                               "شارع رئيسي", "U-Turn",                                          # main
                               "شارع مجمع", "دوار", "طريق خدمة"]}}                              # collector
# ---- C (المحلات): محل أو أرض تجارية قدّام الضلع ≤ 50 م (اختيار الفريق) ----
SHOP_R = 50
COMM_PARCEL = ["تجاري", "متعددة الاستخدام"]
SHOP_GROUPS = {   # نفس مجموعات Overture في كود الاستخدامات السابق (landuse_features_colab.py)
    "auto":       r"car_wash|auto|automotive|gas_station|fuel|mechanic|tire|car_repair|oil_change|car_dealer",
    "nuisance":   r"cafe|coffee|restaurant|fast_food|food_court|burger|pizza|shawarma|juice|dessert|"
                  r"hookah|event|wedding|banquet|gym|fitness|sports_club|stadium|entertainment|amusement|cinema",
    "commercial": r"pharmacy|grocery|supermarket|convenience|market|retail|shop|store|bank|atm|office|"
                  r"salon|barber|laundry|clinic|dentist|doctor|bakery|mall|tailor|optic|electronics"}
FRONT = 1000      # طول المنطقة قدّام الضلع للمسافات dist_major / dist_shop (أبعد = 1000)
CORNER_GAP = 0.5  # نقصّر المنطقة من طرفي الضلع (م)

# ---- أعمدة B للقطعة (نفس كود الضوضاء السابق — Helbich et al. 2025 جدول 1) ----
RAW_CLASSES = {"motorway": ["طريق سريع"], "arterial": ["طريق رئيسي"], "main": ["شارع رئيسي"],
               "collector": ["شارع مجمع"], "local": ["شارع فرعي"], "footway": ["ممر مشاه", "ممر"]}
LEN_BUFFERS = {"motorway": [100, 200, 300, 500, 1000], "arterial": [100, 200, 300, 500, 1000],   # بعد حذف المكرر
               "main": [50, 100, 200, 300, 500, 1000], "collector": [50, 100, 200, 300, 500, 1000],
               "local": [50, 100, 200, 300, 500, 1000], "footway": [50, 100, 200]}            # = 31
BLDG_BUFFERS = [50, 100, 200, 300, 500, 1000]
MAXD = 1000       # المسافة لأقرب طريق: أبعد من 1000 م → 1000

def house_axis(E):
    """اتجاه البيت = أصغر مستطيل يحيط فيه (SMBR) — Duchêne et al. 2003"""
    R_ = np.asarray(shapely.minimum_rotated_rectangle(E).exterior.coords)
    (x0, y0), (x1, y1) = R_[0], R_[1]
    return np.degrees(np.arctan2(x1 - x0, y1 - y0)) % 90

def wall_dir(nz, axis):
    snapped = axis + np.round((np.asarray(nz) - axis) / 45) * 45
    return [dir8(a % 360) for a in snapped]

# ======================= البيانات =======================
d = pd.read_csv(FILE)
lat = next(c for c in d.columns if c.lower() in ("lat", "latitude"))
lon = next(c for c in d.columns if c.lower() in ("lon", "lng", "longitude"))
TO_M = Transformer.from_crs(4326, 32638, always_xy=True)

parcels = gpd.read_parquet(f"{BASE}/riyadh_parcels_all.parquet").to_crs(32638)
PA_G = shapely.make_valid(parcels.geometry.to_numpy()); PA_TREE = shapely.STRtree(PA_G)
PA_SUB = parcels["ParcelSub"].astype(str).to_numpy(object) if "ParcelSub" in parcels else np.array([""] * len(parcels), object)
PA_COMM = np.isin(PA_SUB, COMM_PARCEL)

streets = gpd.read_parquet(f"{BASE}/riyadh_streets.parquet").to_crs(32638)
wcol = next((c for c in streets.columns if "width" in c.lower() or "عرض" in c), None)
ccol = next((c for c in streets.columns if c.upper() in ("SUBTYPECLASS", "CLASS", "STREETCLASS") or "تصنيف" in c), None)
print("أعمدة الشوارع — العرض:", wcol, "| النوع:", ccol)
streets["W"] = pd.to_numeric(streets[wcol], errors="coerce") if wcol else np.nan
streets["W"] = streets["W"].where(streets["W"] > 0)
if ccol: streets["W"] = streets["W"].fillna(streets.groupby(ccol)["W"].transform("median"))
streets["W"] = streets["W"].fillna(streets["W"].median())
ST_G = streets.geometry.to_numpy(); ST_W = streets["W"].to_numpy(float)
ST_C = streets[ccol].astype(str).to_numpy(object) if ccol else np.array([""] * len(streets), object)
ST_TREE = shapely.STRtree(ST_G)

ST_CLS = streets[ccol].astype(str).to_numpy(object)
ST_LIM = np.array([R_LIMIT.get(c, 0) for c in ST_CLS], float)           # 0 = ما ينحسب في R
print("تصنيفات R:", {c: R_LIMIT[c] for c in R_LIMIT if c in set(ST_CLS)})
RAW = {}
for k, subs in RAW_CLASSES.items():
    m = np.isin(ST_CLS, subs)
    g = ST_G[m]; RAW[k] = (g, ST_W[m], shapely.STRtree(g))

gba = pd.concat([gpd.read_parquet(f"{BASE}/GBA/{t}.parquet")
                 for t in ["e045_n25_e050_n20", "e045_n30_e050_n25"]]).to_crs(32638)
GBA_G = gba.geometry.to_numpy(); GBA_TREE = shapely.STRtree(GBA_G)

pl = gpd.read_parquet(f"{BASE}/overture_places_riyadh.parquet")
def _first(v, key):
    if isinstance(v, dict): return v.get(key)
    if isinstance(v, str):
        try: return json.loads(v).get(key)
        except Exception: return None
    return None
pl["cat"] = pl["categories"].map(lambda v: _first(v, "primary")) if "categories" in pl else None
if "basic_category" in pl: pl["cat"] = pl["cat"].fillna(pl["basic_category"])
pl["cat"] = pl["cat"].fillna("").astype(str).str.lower()
pat = "|".join(f"(?:{p})" for p in SHOP_GROUPS.values())
pl = pl[pl["cat"].str.contains(pat, regex=True)].to_crs(32638)
SH_G = pl.geometry.to_numpy(); SH_TREE = shapely.STRtree(SH_G)
print(f"محلات Overture: {len(pl):,} | قطع تجارية/متعددة: {PA_COMM.sum():,}")

# ======================= 1) القطعة + البيت =======================
def get_parcel(la, lo):
    """نفس كود الداتا سيت، ويرجّع رقم القطعة بعد (عشان ما نحسب قطعتنا تجارية)"""
    pt = Point(*TO_M.transform(lo, la))
    idx = PA_TREE.query(pt, predicate="intersects")
    if len(idx):
        res = np.array(["سكني" in str(PA_SUB[i]) for i in idx])
        if res.any(): idx = idx[res]
        j = idx[np.argmin(shapely.area(PA_G[idx]))]; return PA_G[j], j
    idx = PA_TREE.query(pt.buffer(30), predicate="intersects")
    if not len(idx): return None, None
    j = idx[np.argmin(shapely.distance(PA_G[idx], pt))]; return PA_G[j], j

def real_house(P):
    """نفس كود الداتا سيت — ويرجّع أرقام مباني GBA اللي تكوّن البيت (عشان ما نحسبها جيران)"""
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    idx = GBA_TREE.query(P.buffer(5), predicate="intersects")
    if len(idx) == 0: return None, None
    g = GBA_G[idx]
    pc = P.centroid if P.contains(P.centroid) else P.representative_point()
    area = shapely.area(g); inter = shapely.area(shapely.intersection(g, P))
    frac = np.divide(inter, area, out=np.zeros_like(area), where=area > 0)
    site = shapely.within(shapely.centroid(g), P) | (frac >= 0.5) | shapely.contains(g, pc) \
           | (shapely.distance(g, pc) <= 2.0)
    if not site.any(): return None, None
    H = shapely.union_all(g[site]).intersection(P.buffer(1.0))
    if H.geom_type == "MultiPolygon": H = max(H.geoms, key=lambda q: q.area)
    return (H, idx[site]) if (not H.is_empty and H.area >= MIN_AREA) else (None, None)

# ======================= 1ب) أضلاع القطعة وشوارعها =======================
def parcel_sides(P):
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    ring = np.asarray(orient(P, 1.0).exterior.coords); sides = []
    for (x1, y1), (x2, y2) in zip(ring[:-1], ring[1:]):
        dx, dy = x2 - x1, y2 - y1; L = np.hypot(dx, dy)
        if L < 0.5: continue
        seg = LineString([(x1, y1), (x2, y2)]); mid = seg.interpolate(0.5, normalized=True)
        nz = np.degrees(np.arctan2(dy, -dx)) % 360; edir = np.degrees(np.arctan2(dx, dy)) % 180
        street = None
        idx = ST_TREE.query(mid.buffer(CAND_RADIUS))
        if len(idx):
            resid = shapely.distance(ST_G[idx], mid) - ST_W[idx] / 2
            for j in idx[np.argsort(np.abs(resid))]:
                if shapely.distance(ST_G[j], mid) - ST_W[j] / 2 > TOL: break
                par = adiff(line_dir(ST_G[j], mid), edir)
                if min(par, 180 - par) <= MAX_ANGLE: street = j; break
        sides.append(dict(seg=seg, nz=nz, dir=dir8(nz), length=L, street=street,
                          width=ST_W[street] if street is not None else np.nan,
                          cls=ST_C[street] if street is not None else None))
    return P, sides

# ======================= 1ج) الغلاف (للفاضية) =======================
def setback(width, cls, is_main):
    """اشتراطات 4-1-4 و 4-1-5 (صفحة 17)"""
    cls = cls if isinstance(cls, str) else ""
    w = width if pd.notna(width) and width > 0 else 0
    if "ممر" in cls:                  return 1.5
    if "ساح" in cls or "ميد" in cls:  return max(3.0, w / 5)
    if is_main:                        return 6.0 if w >= 30 else max(3.0, w / 5)
    return max(2.0, w / 5)

FALLBACK_R = 50.0                                   # أبعد مسافة نبحث فيها عن شارع قدّام الضلع (م)
def nearest_front_street(sides):
    """لو ما فيه شارع ملاصق: نختار الضلع اللي قدّامه أقرب شارع (حافة الشارع، مو خط المنتصف)"""
    best = None
    for s in sides:
        mid = s["seg"].interpolate(0.5, normalized=True)
        nx, ny = np.sin(np.radians(s["nz"])), np.cos(np.radians(s["nz"]))
        idx = ST_TREE.query(mid.buffer(FALLBACK_R))
        for j in idx:
            q = ST_G[j].interpolate(ST_G[j].project(mid))               # أقرب نقطة على الشارع
            vx, vy = q.x - mid.x, q.y - mid.y; dist = np.hypot(vx, vy)
            if dist > 0 and (vx * nx + vy * ny) / dist < np.cos(np.radians(60)):
                continue                                                # الشارع مو قدّام الضلع
            edge = max(dist - ST_W[j] / 2, 0)
            if edge <= FALLBACK_R and (best is None or edge < best[0]):
                best = (edge, s, j)
    if best is None: return False
    _, s, j = best
    s.update(street=j, width=ST_W[j], cls=ST_C[j])
    return True

def envelope(P, sides):
    st = [s for s in sides if s["street"] is not None]
    if not st and nearest_front_street(sides):                          # تصحيح: أقرب شارع قدّام القطعة
        st = [s for s in sides if s["street"] is not None]
    main = max(st, key=lambda s: (s["width"], s["length"])) if st else None
    cut, nb = [], []
    for s in sides:
        if s["street"] is not None:
            s["setback"] = setback(s["width"], s["cls"], s is main)
            s["type"] = "main street" if s is main else "street"
        else:
            s["setback"], s["type"] = NB_SETBACK, "neighbour"; nb.append(s["seg"])
        cut.append(s["seg"].buffer(s["setback"], cap_style=3))
    E = P.difference(shapely.union_all(cut))
    if E.geom_type == "MultiPolygon": E = max(E.geoms, key=lambda g: g.area)
    if E.is_empty or E.area < 0.2 * P.area: E = P
    if E.area > COVER_MAX * P.area and nb:                              # > 75%: نزيد من جهة الجيران
        nbu = shapely.union_all(nb); lo, hi = NB_SETBACK, NB_SETBACK + 10
        for _ in range(25):
            mid = (lo + hi) / 2
            Em = E.difference(nbu.buffer(mid, cap_style=3))
            Em = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
            lo, hi = (mid, hi) if Em.area > COVER_MAX * P.area else (lo, mid)
        Em = E.difference(nbu.buffer(hi, cap_style=3))
        E = max(Em.geoms, key=lambda g: g.area) if Em.geom_type == "MultiPolygon" else Em
        for s in sides:
            if s["type"] == "neighbour": s["setback"] = round(hi, 2)
    return E, main


def house_of(P):
    E, site_idx = real_house(P)
    if E is not None: return E, site_idx
    P1, sides = parcel_sides(P)
    E, _ = envelope(P1, sides)
    return E, GBA_TREE.query(E, predicate="intersects")

# ======================= 2) لكل ضلع: R و C والمسافات =======================
def edges(poly, min_len):
    """أضلاع الشكل: (بداية، نهاية، العمود للخارج، الطول)"""
    ring = np.asarray(orient(poly, 1.0).exterior.coords); out = []
    for (x1, y1), (x2, y2) in zip(ring[:-1], ring[1:]):
        dx, dy = x2 - x1, y2 - y1; L = np.hypot(dx, dy)
        if L < min_len: continue
        out.append(((x1, y1), (x2, y2), dy / L, -dx / L, L))
    return out

def side_noise(P, pid, a, b, nx, ny):
    seg = LineString([a, b])
    # المنطقة قدّام الضلع — نقصّرها 0.5 م من كل طرف عشان القطعة اللي تلمس الزاوية بس
    # (جارة الضلع اللي جنبه) ما تنحسب قدّام هالضلع
    L = seg.length; s = min(CORNER_GAP, L / 4); ux, uy = (b[0] - a[0]) / L, (b[1] - a[1]) / L
    a2, b2 = (a[0] + ux * s, a[1] + uy * s), (b[0] - ux * s, b[1] - uy * s)
    C_ = Polygon([a2, b2, (b2[0] + nx * FRONT, b2[1] + ny * FRONT), (a2[0] + nx * FRONT, a2[1] + ny * FRONT)])
    # --- الشوارع قدّام الضلع: المسافة لحافة الشارع ---
    R, dmaj = 0, MAXD
    idx = ST_TREE.query(C_, predicate="intersects")
    idx = idx[ST_LIM[idx] > 0]
    if len(idx):
        part = shapely.intersection(ST_G[idx], C_)
        dist = np.clip(shapely.distance(part, seg) - ST_W[idx] / 2, 0, None)
        R = int((dist <= ST_LIM[idx]).any())
        dmaj = float(min(dist.min(), MAXD))
    # --- المحلات + الأراضي التجارية قدّام الضلع ---
    dshop = MAXD
    idx = SH_TREE.query(C_, predicate="intersects")
    if len(idx): dshop = min(dshop, float(shapely.distance(SH_G[idx], seg).min()))
    idx = PA_TREE.query(C_, predicate="intersects")
    idx = idx[PA_COMM[idx] & (idx != pid)]
    if len(idx): dshop = min(dshop, float(shapely.distance(shapely.intersection(PA_G[idx], C_), seg).min()))
    C = int(dshop <= SHOP_R)
    return R, C, dmaj, dshop

# ======================= 3) حساب صف واحد =======================
def row_noise(i):
    P, pid = get_parcel(d.at[i, lat], d.at[i, lon])
    if P is None: return None, None
    if P.geom_type == "MultiPolygon": P = max(P.geoms, key=lambda g: g.area)
    E, site_idx = house_of(P)
    ax = house_axis(E); out = {}

    # --- لكل جهة: أضلاع القطعة (الضوضاء) ---
    rows = []
    for a, b, nx, ny, L in edges(P, 0.5):
        R, C, dmaj, dshop = side_noise(P, pid, a, b, nx, ny)
        rows.append(dict(dir=wall_dir([np.degrees(np.arctan2(nx, ny)) % 360], ax)[0], R=R, C=C,
                         dmaj=dmaj, dshop=dshop, a=a, b=b, nx=nx, ny=ny))
    S = pd.DataFrame(rows)
    for x, g in S.groupby("dir"):
        out[f"Noise_{x}"] = int(g.R.max() + g.C.max())                      # R + C (0–2)
        out[f"dist_major_{x}"] = round(g.dmaj.min(), 1)
        out[f"dist_shop_{x}"] = round(g.dshop.min(), 1)

    # --- لكل جهة: طول جدران البيت ---
    W = pd.DataFrame([dict(dir=wall_dir([np.degrees(np.arctan2(nx, ny)) % 360], ax)[0], L=L)
                      for a, b, nx, ny, L in edges(E, 1.0)])
    for x, g in W.groupby("dir"): out[f"facade_length_{x}"] = round(g.L.sum(), 2)

    # --- للقطعة: المسافة لأقرب طريق + أطوال الطرق داخل النطاقات ---
    rings = {r: P.buffer(r) for r in BLDG_BUFFERS}
    for k, (g, w, t) in RAW.items():
        idx = t.query(rings[1000])
        if len(idx):
            gi = g[idx]
            out[f"dist_road_{k}"] = round(float(min(np.clip(shapely.distance(gi, P) - w[idx] / 2, 0, None).min(), MAXD)), 1)
        else:
            out[f"dist_road_{k}"] = float(MAXD)
        for r in LEN_BUFFERS[k]:
            out[f"len_{k}_{r}m"] = round(float(shapely.length(shapely.intersection(g[idx], rings[r])).sum()), 1) if len(idx) else 0.0

    # --- للقطعة: عدد المباني حولها (بدون البيت نفسه) ---
    idx = GBA_TREE.query(rings[1000]); idx = idx[~np.isin(idx, site_idx)]
    for r in BLDG_BUFFERS:
        out[f"n_bldg_{r}m"] = int(shapely.intersects(GBA_G[idx], rings[r]).sum()) if len(idx) else 0
    return out, (P, E, S)

COLS = ([f"{p}_{x}" for p in ["Noise", "dist_major", "dist_shop", "facade_length"] for x in DIRS]
        + [f"dist_road_{k}" for k in RAW_CLASSES]
        + [f"len_{k}_{r}m" for k in RAW_CLASSES for r in LEN_BUFFERS[k]]
        + [f"n_bldg_{r}m" for r in BLDG_BUFFERS])
print("عدد الأعمدة:", len(COLS))

# ======================= 4أ) تجربة =======================
if TEST:
    done = 0
    for i in d.sample(frac=1).index:
        if done == 3: break
        out, info = row_noise(i)
        if not out: continue
        done += 1; P, E, S = info
        print(f"\n========== القطعة {done}  ({d.at[i, lat]:.6f}, {d.at[i, lon]:.6f}) ==========")
        T = pd.DataFrame({x: {"Noise": out.get(f"Noise_{x}"), "dist_major": out.get(f"dist_major_{x}"),
                              "dist_shop": out.get(f"dist_shop_{x}"), "facade_length": out.get(f"facade_length_{x}")}
                          for x in DIRS}).T.dropna(how="all")
        T.index.name = "الجهة"; print(T.to_string())
        print("للقطعة:", {k: v for k, v in out.items() if k.startswith(("dist_road", "n_bldg"))})
        print(f"https://www.google.com/maps?q={d.at[i, lat]},{d.at[i, lon]}&t=k")

        fig, ax = plt.subplots(figsize=(8, 8)); V = P.buffer(150)
        for j in ST_TREE.query(V):
            col = "#d62728" if ST_LIM[j] == 500 else "#ff7f0e" if ST_LIM[j] == 100 else "#c7c7c7"
            gpd.GeoSeries([ST_G[j].buffer(ST_W[j] / 2, cap_style=2)]).plot(ax=ax, color=col, alpha=.5, zorder=1)
        cm = PA_TREE.query(V); cm = cm[PA_COMM[cm]]
        if len(cm): gpd.GeoSeries(list(PA_G[cm])).plot(ax=ax, facecolor="none", edgecolor="#9467bd", hatch="//", lw=.6, zorder=2)
        sh = SH_TREE.query(V)
        if len(sh): gpd.GeoSeries(list(SH_G[sh])).plot(ax=ax, color="#9467bd", markersize=14, zorder=4)
        gpd.GeoSeries([P]).plot(ax=ax, facecolor="none", edgecolor="k", lw=1.5, zorder=3)
        gpd.GeoSeries([E]).plot(ax=ax, color="#f3efe6", edgecolor="#555", zorder=3)
        for _, s in S.iterrows():
            m = ((s.a[0] + s.b[0]) / 2, (s.a[1] + s.b[1]) / 2)
            ax.text(m[0] + s.nx * 9, m[1] + s.ny * 9, f"{s.dir}\nR{s.R}+C{s.C}", ha="center", va="center", fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="k", lw=.5), zorder=6)
        x0, y0, x1, y1 = P.buffer(120).bounds; ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        ax.annotate("N", xy=(.95, .95), xytext=(.95, .85), xycoords="axes fraction", ha="center", arrowprops=dict(arrowstyle="->", lw=1.5))
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color="#d62728", alpha=.5, label="road: 500 m class"), Patch(color="#ff7f0e", alpha=.5, label="road: 100 m class"),
                           Patch(color="#c7c7c7", label="local / not counted"), Patch(facecolor="none", edgecolor="#9467bd", hatch="//", label="commercial parcel"),
                           Patch(color="#9467bd", label="shop (Overture)")], loc="lower left", fontsize=8)
        ax.set_title(f"Parcel {done} — Noise = R + C per side", fontsize=10); plt.show()

# ======================= 4ب) كل الملف + الحفظ المرحلي =======================
else:
    for c in COLS:
        if c not in d: d[c] = np.nan
    def save():
        tmp = FILE + ".tmp"
        d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)
    todo = d.index[d["n_bldg_50m"].isna()]                               # يكمّل من وين وقف
    print(f"باقي {len(todo)} من {len(d)} صف")
    errors, t0 = 0, time.time()
    for n, i in enumerate(todo, 1):
        try:
            out, _ = row_noise(i)
            if not out: errors += 1
            else:
                for k, v in out.items(): d.at[i, k] = v
        except Exception as e:
            errors += 1; print(f"  صف {i}: {type(e).__name__}: {e}")
        if n % SAVE_EVERY == 0:
            save(); el = time.time() - t0
            print(f"  حُفظ {n}/{len(todo)} — باقي تقريباً {el / n * (len(todo) - n) / 60:.0f} دقيقة")
    save()
    print(f"\n✅ خلص — صفوف ما انحسبت: {errors}")
    print("\nتوزيع الضوضاء لكل جهة:")
    print(pd.DataFrame({x: d[f"Noise_{x}"].value_counts().reindex([0, 1, 2]) for x in DIRS}).fillna(0).astype(int).to_string())
    print("\nالمسافات (م):"); print(d[[c for c in COLS if c.startswith(("dist_major", "dist_shop"))]].describe().round(1).T[["mean", "50%", "min", "max"]].to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
أعمدة الشوارع — العرض: STREETWIDTH | النوع: STREETCLASS
تصنيفات R: {'طريق سريع': 500, 'مخرج': 500, 'مدخل': 500, 'طريق رئيسي': 100, 'كوبري': 100, 'شارع رئيسي': 100, 'U-Turn': 100, 'شارع مجمع': 100, 'دوار': 100, 'طريق خدمة': 100}
محلات Overture: 24,179 | قطع تجارية/متعددة: 198,118
عدد الأعمدة: 75
باقي 5790 من 5790 صف
  حُفظ 100/5790 — باقي تقريباً 3 دقيقة
  حُفظ 200/5790 — باقي تقريباً 3 دقيقة
  حُفظ 300/5790 — باقي تقريباً 3 دقيقة
  حُفظ 400/5790 — باقي تقريباً 3 دقيقة
  حُفظ 500/5790 — باقي تقريباً 3 دقيقة
  حُفظ 600/5790 — باقي تقريباً 3 دقيقة
  حُفظ 700/5790 — باقي تقريباً 3 دقيقة
  حُفظ 800/5790 — باقي تقريباً 3 دقيقة
  حُفظ 900/5790 — باقي تقريباً 3 دقيقة
  حُفظ 1000/5790 — باقي تقريباً 3 دقيقة
  حُفظ 1100/5790 — باقي تقريباً 2 دقيقة
  حُفظ 1200/5790 — باقي تقريباً 2 دقيقة
  حُفظ 1300/5790 — باقي تقريباً 2 دقيقة
  حُفظ 1400/5790 — باقي تقريباً 2 دقيقة
  ح

In [ ]:
STUDENT = "deema"
import os, pandas as pd
FILE = f"/content/drive/MyDrive/Maamar/villas_{STUDENT}.csv"
d = pd.read_csv(FILE)
d = d.drop(columns=["height_source", "parcel_wkt"], errors="ignore")
tmp = FILE + ".tmp"; d.to_csv(tmp, index=False, encoding="utf-8-sig"); os.replace(tmp, FILE)
print("✅ الأعمدة:", len(d.columns))

/tmp/ipykernel_592/1155179846.py:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_csv(FILE)


✅ الأعمدة: 138


# Add Files

In [1]:
# ============================================================
# pois_riyadh_merged.parquet — نفس الخلية 60 في Preparing_dataset بالضبط، بس لحالها وتنحفظ في Drive: Maamar
# دمج الأنشطة من 3 مصادر مفتوحة ومجانية (بدون API ولا مفتاح):
#   1) Overture Maps Places   (overture_places_riyadh.parquet اللي في Maamar)
#   2) OpenStreetMap          (ملف دول الخليج من Geofabrik)
#   3) Foursquare OS Places   (نسخة Source Cooperative العامة)
# ثم حذف المكرر: نفس المجموعة + مصدر مختلف + أقرب من 20 م = نفس المكان
# الفرق الوحيد: النطاق كان من حدود قطع الداتا سيت (± 0.01°)؛ هنا من حدود ملف Overture
# اللي تنزّل بنفس النطاق، فيطلع نفسه تقريباً بدون ما نحتاج ملف الداتا سيت.
# ============================================================
!apt-get -qq install -y osmium-tool > /dev/null
!pip -q install duckdb boto3

import os, re, json, subprocess
import numpy as np, pandas as pd, geopandas as gpd, shapely, duckdb, boto3
from botocore import UNSIGNED
from botocore.config import Config
from google.colab import drive

drive.mount("/content/drive")
BASE   = "/content/drive/MyDrive/Maamar"
PLACES = f"{BASE}/overture_places_riyadh.parquet"
MERGED = f"{BASE}/pois_riyadh_merged.parquet"
DEDUP_M = 20       # م: نفس المجموعة وأقرب من هذا بين مصدرين = نفس المكان (يُحسب مرة)

ov_raw = gpd.read_parquet(PLACES).to_crs(4326)
W, S, E, N = ov_raw.total_bounds
print(f"النطاق: {W:.4f}, {S:.4f}, {E:.4f}, {N:.4f}")

def _first(v, key):
    if isinstance(v, dict):
        return v.get(key)
    if isinstance(v, str):
        try: return json.loads(v).get(key)
        except Exception: return None
    return None

# ---------- المجموعات (مصححة) ----------
GROUPS = [
    ("mosque",     r"mosque|muslim|islamic|place_of_worship|religious"),
    ("education",  r"school|kindergarten|preschool|nursery|day_care|daycare|college|university|education|place_of_learning"),
    ("hospital",   r"hospital|emergency"),
    ("government", r"government|public_service|police|court|embassy|municipal|post_office|townhall|city_hall"),
    ("park",       r"^(park|playground|garden|botanical_garden|recreation_area|plaza|dog_park|leisure_park|leisure_playground|leisure_garden)$"),
    ("auto",       r"car_wash|automotive|auto_|^auto|gas_station|fuel|mechanic|tire|car_repair|oil_change|car_dealer|shop_car"),
    ("nuisance",   r"caf[eé]|coffee|restaurant|eatery|fast_food|food_court|burger|pizza|shawarma|juice|dessert|"
                   r"cafeteria|tea_house|hookah|shisha|event|wedding|banquet|gym|fitness|sports|stadium|"
                   r"entertainment|amusement|cinema|leisure_sports_centre|leisure_fitness_centre"),
    ("commercial", r"pharmacy|grocery|supermarket|convenience|market|retail|shop|store|bank|atm|office|"
                   r"salon|barber|laundry|clinic|dentist|doctor|bakery|mall|tailor|optic|electronics"),
]
GROUP_NAMES = [g for g, _ in GROUPS]
def group_of(c):
    if not c or "parking" in c:
        return None
    for g, pat in GROUPS:
        if re.search(pat, c):
            return g
    return None

parts = []

# ---------- 1) Overture ----------
cat = ov_raw["categories"].map(lambda v: _first(v, "primary")) if "categories" in ov_raw.columns else ov_raw["basic_category"]
ov = gpd.GeoDataFrame({"src": "overture",
                       "name": ov_raw["names"].map(lambda v: _first(v, "primary")) if "names" in ov_raw.columns else None,
                       "cat": cat.fillna("").astype(str).str.lower()}, geometry=ov_raw.geometry, crs=ov_raw.crs)
parts.append(ov); print(f"Overture: {len(ov):,}")

# ---------- 2) OpenStreetMap (Geofabrik) ----------
OSM_OUT = f"{BASE}/osm_pois_riyadh.parquet"
if not os.path.exists(OSM_OUT):
    pbf = "/content/gcc-states-latest.osm.pbf"
    if not os.path.exists(pbf):
        print("⬇️ تنزيل OSM (دول الخليج) من Geofabrik ...")
        subprocess.run(["wget", "-q", "-O", pbf, "https://download.geofabrik.de/asia/gcc-states-latest.osm.pbf"], check=True)
    subprocess.run(["osmium", "extract", "-b", f"{W},{S},{E},{N}", pbf, "-o", "/content/riyadh.osm.pbf", "--overwrite"], check=True)
    subprocess.run(["osmium", "tags-filter", "/content/riyadh.osm.pbf", "nwr/amenity", "nwr/shop", "nwr/leisure",
                    "nwr/office", "-o", "/content/riyadh_poi.osm.pbf", "--overwrite"], check=True)
    subprocess.run(["osmium", "export", "/content/riyadh_poi.osm.pbf", "-f", "geojsonseq",
                    "-o", "/content/riyadh_poi.geojsonseq", "--overwrite"], check=True)
    rows = []
    with open("/content/riyadh_poi.geojsonseq", encoding="utf-8") as f:
        for line in f:
            ft = json.loads(line.strip().lstrip("\x1e"))
            t = ft.get("properties", {})
            a, sh, le, of = t.get("amenity"), t.get("shop"), t.get("leisure"), t.get("office")
            if a == "place_of_worship":
                c = "mosque" if t.get("religion") == "muslim" else "place_of_worship"
            elif a:  c = a
            elif sh: c = f"shop_{sh}"
            elif le: c = f"leisure_{le}"
            elif of: c = "government_office" if of == "government" else f"office_{of}"
            else:    continue
            g = shapely.geometry.shape(ft["geometry"])
            rows.append({"name": t.get("name:ar") or t.get("name") or t.get("name:en"), "cat": c.lower(),
                         "geometry": g if g.geom_type == "Point" else g.representative_point()})
    gpd.GeoDataFrame(rows, geometry="geometry", crs=4326).to_parquet(OSM_OUT)
osm = gpd.read_parquet(OSM_OUT); osm["src"] = "osm"
parts.append(osm); print(f"OSM: {len(osm):,}")

# ---------- 3) Foursquare OS Places (Source Cooperative، عام بدون حساب) ----------
FSQ_OUT = f"{BASE}/fsq_os_places_riyadh.parquet"
if not os.path.exists(FSQ_OUT):
    s3 = boto3.client("s3", endpoint_url="https://data.source.coop", config=Config(signature_version=UNSIGNED))
    pref = s3.list_objects_v2(Bucket="fused", Prefix="fsq-os-places/", Delimiter="/").get("CommonPrefixes", [])
    rel = sorted(p["Prefix"] for p in pref if re.search(r"\d{4}-\d{2}-\d{2}/$", p["Prefix"]))[-1]
    print(f"⬇️ Foursquare OS Places — الإصدار: {rel}")
    con = duckdb.connect()
    con.sql("INSTALL httpfs; LOAD httpfs;")
    con.sql("SET s3_endpoint='data.source.coop'; SET s3_url_style='path'; SET s3_region='us-west-2';")
    fsq = con.sql(f"""
        SELECT name, latitude, longitude, CAST(fsq_category_labels AS VARCHAR) AS labels
        FROM read_parquet('s3://fused/{rel}places/*.parquet')
        WHERE country = 'SA' AND date_closed IS NULL
          AND longitude BETWEEN {W} AND {E} AND latitude BETWEEN {S} AND {N}
    """).df()
    leaf = fsq["labels"].fillna("").str.split(">").str[-1].str.strip(" []'\"").str.lower().str.replace(r"[ ,&/-]+", "_", regex=True)
    fsq = gpd.GeoDataFrame({"name": fsq["name"], "cat": leaf},
                           geometry=gpd.points_from_xy(fsq["longitude"], fsq["latitude"]), crs=4326)
    fsq.to_parquet(FSQ_OUT)
fsq = gpd.read_parquet(FSQ_OUT); fsq["src"] = "fsq"
parts.append(fsq); print(f"Foursquare OS: {len(fsq):,}")

# ---------- الدمج وحذف المكرر ----------
allpoi = pd.concat([p[["src", "name", "cat", "geometry"]].to_crs(32638) for p in parts], ignore_index=True)
allpoi = gpd.GeoDataFrame(allpoi, geometry="geometry", crs=32638)
allpoi["grp"] = allpoi["cat"].map({c: group_of(c) for c in allpoi["cat"].unique()})
allpoi = allpoi[allpoi["grp"].notna()].reset_index(drop=True)

keep = np.ones(len(allpoi), bool)
order = allpoi["src"].map({"fsq": 0, "osm": 1, "overture": 2}).to_numpy()
G = allpoi.geometry.to_numpy(); GR = allpoi["grp"].to_numpy(object); SRC = allpoi["src"].to_numpy(object)
T = shapely.STRtree(G)
a, b = T.query(G, predicate="dwithin", distance=DEDUP_M)
m = (a < b) & (GR[a] == GR[b]) & (SRC[a] != SRC[b])          # نفس المجموعة، مصدر مختلف، أقرب من 20 م
for i, j in zip(a[m], b[m]):
    if keep[i] and keep[j]:
        keep[j if order[j] >= order[i] else i] = False          # نخلي نسخة المصدر الأعلى (FSQ ثم OSM ثم Overture)
merged = allpoi[keep].reset_index(drop=True)
merged.to_parquet(MERGED)

print(f"\nقبل حذف المكرر: {len(allpoi):,} | بعد: {len(merged):,}")
print(f"💾 انحفظ: {MERGED}")
display(pd.crosstab(merged["grp"], merged["src"], margins=True))

Mounted at /content/drive
النطاق: 46.3011, 24.4014, 47.0810, 25.0938
Overture: 45,433
⬇️ تنزيل OSM (دول الخليج) من Geofabrik ...
OSM: 21,984
⬇️ Foursquare OS Places — الإصدار: fsq-os-places/2025-02-06/


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Foursquare OS: 173,475

قبل حذف المكرر: 138,369 | بعد: 117,513
💾 انحفظ: /content/drive/MyDrive/Maamar/pois_riyadh_merged.parquet


src,fsq,osm,overture,All
grp,,,,
auto,4949,706,768,6423
commercial,40453,1618,4246,46317
education,7725,911,1019,9655
government,2242,330,388,2960
hospital,2357,130,624,3111
mosque,2179,3161,189,5529
nuisance,34019,812,4861,39692
park,1964,1653,209,3826
All,95888,9321,12304,117513


In [2]:
# ============================================================
# riyadh_transit.parquet — مواقف الباص ومحطات وخطوط المترو في الرياض، من OpenStreetMap (Geofabrik)
# تنحفظ في Drive: Maamar، والموقع يرسمها في مخطط الطرق بدل ما يسأل OpenStreetMap وقت التحليل.
# الأعمدة: kind (bus = موقف باص، metro = محطة مترو/قطار، line = خط مترو/قطار)، railway، name، geometry (EPSG:4326)
# المصدر: © OpenStreetMap contributors (ODbL) — لازم يُذكر على المخطط
# ============================================================
!apt-get -qq install -y osmium-tool > /dev/null

import os, json, subprocess
import geopandas as gpd, shapely
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Maamar"
OUT  = f"{BASE}/riyadh_transit.parquet"
W, S, E, N = 46.15, 24.25, 47.35, 25.25        # حدود الرياض الثابتة (نفس tools/clip_gba.py)

# ---------- تنزيل OSM وقصّه على الرياض ----------
pbf = "/content/gcc-states-latest.osm.pbf"
if not os.path.exists(pbf):
    print("⬇️ تنزيل OSM (دول الخليج) من Geofabrik ...")
    subprocess.run(["wget", "-q", "-O", pbf, "https://download.geofabrik.de/asia/gcc-states-latest.osm.pbf"], check=True)
subprocess.run(["osmium", "extract", "-b", f"{W},{S},{E},{N}", pbf, "-o", "/content/riyadh.osm.pbf", "--overwrite"], check=True)

# ---------- المواقف والمحطات والخطوط ----------
subprocess.run(["osmium", "tags-filter", "/content/riyadh.osm.pbf",
                "n/highway=bus_stop", "n/public_transport=station,stop_position,platform",
                "n/railway=station,halt,subway_entrance,tram_stop",
                "w/railway=subway,light_rail,rail,monorail,tram",
                "-o", "/content/transit.osm.pbf", "--overwrite"], check=True)
subprocess.run(["osmium", "export", "/content/transit.osm.pbf", "-f", "geojsonseq",
                "-o", "/content/transit.geojsonseq", "--overwrite"], check=True)

rows = []
with open("/content/transit.geojsonseq", encoding="utf-8") as f:
    for line in f:
        ft = json.loads(line.strip().lstrip("\x1e"))
        t = ft.get("properties", {})
        g = shapely.geometry.shape(ft["geometry"])
        name = t.get("name:ar") or t.get("name") or t.get("name:en")
        if g.geom_type == "Point":
            rw, pt = t.get("railway"), t.get("public_transport")
            if t.get("highway") == "bus_stop" or (pt in ("stop_position", "platform") and t.get("bus") == "yes"):
                kind = "bus"
            elif rw in ("station", "halt", "subway_entrance", "tram_stop") or (pt == "station" and t.get("bus") != "yes"):
                kind = "metro"
            elif pt == "station":
                kind = "bus"                       # محطة باصات
            else:
                continue                           # نقاط الخطوط نفسها (تقاطعات وغيرها) ما نبيها
        elif g.geom_type in ("LineString", "MultiLineString"):
            if t.get("railway") not in ("subway", "light_rail", "rail", "monorail", "tram"):
                continue
            kind = "line"
        else:
            continue
        rows.append({"kind": kind, "railway": t.get("railway"), "name": name, "geometry": g})

tr = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326)
# نفس الموقف ممكن ينرسم مرتين في OSM (نقطة الموقف + نقطة الوقوف): نخلي وحدة لكل 15 م من نفس النوع
pts = tr[tr["kind"] != "line"].to_crs(32638)
keep = []
for kind, g in pts.groupby("kind"):
    tree_geoms = g.geometry.to_numpy()
    tree = shapely.STRtree(tree_geoms)
    used = set()
    for k, geom in enumerate(tree_geoms):
        if k in used:
            continue
        near = tree.query(geom, predicate="dwithin", distance=15)
        used.update(int(x) for x in near)
        keep.append(g.index[k])
tr = tr.loc[sorted(keep) + list(tr.index[tr["kind"] == "line"])].reset_index(drop=True)

tr.to_parquet(OUT)
print(f"💾 انحفظ: {OUT}")
print(tr["kind"].value_counts().rename({"bus": "مواقف باص", "metro": "محطات مترو/قطار", "line": "خطوط"}).to_string())
print("\nأسماء بعض محطات المترو:", tr.loc[tr["kind"] == "metro", "name"].dropna().unique()[:15].tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
💾 انحفظ: /content/drive/MyDrive/Maamar/riyadh_transit.parquet
kind
خطوط               884
مواقف باص          528
محطات مترو/قطار    323

أسماء بعض محطات المترو: ['محطة سكة حديد الرياض', 'مدخل 1;محطة د.سليمان الحبيب - الأزرق 12', 'مدخل3؛ محطة د.سليمان الحبيب - الأزرق 12', 'مدخل 2;محطة د.سليمان الحبيب - الأزرق 12', 'مدخل4؛ محطة د.سليمان الحبيب - الأزرق 12', 'المروج', 'محطة جامعة الاميره نوره 1', '5 الصالة', '3-4 الصالة', '1-2 الصالة', 'محطة جامعة الاميره نوره 2', 'وزارة التعليم', 'محطة طريق جدة', 'محطة الدوح مدخل 1', 'محطة الدوح مدخل 2']
